In [ ]:
#!pip install xvec
# using xvec https://xvec.readthedocs.io/en/stable/zonal_stats.html
#!pip install easysnowdata --upgrade

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import dask.dataframe as dd
import seaborn as sns
import xarray as xr
import geopandas as gpd
import xvec
import coiled
import dask
from global_snowmelt_runoff_onset.config import Config
import global_snowmelt_runoff_onset.analysis as analysis
from xarray.groupers import BinGrouper
import ee
import flox.xarray
import easysnowdata
import os
import glob
from shapely.geometry import Point
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import Normalize
import matplotlib.colors as colors
import textwrap
import rioxarray
from cartopy import crs as ccrs
from cartopy import feature as cfeature
import easysnowdata
import rioxarray as rxr
#ee.Initialize()

In [2]:
config = Config('config/global_config_v9.txt')

SAS token is valid until 2025-11-11 05:17 UTC (45.0 hours)
----------------------------------------
Configuration loaded:
config_name = global_config_v9
version = v9
resolution = 0.00072000072000072
bands = vv
mountain_snow_only = False
spatial_chunk_dim_s1_read = 2048
spatial_chunk_dim_s1_process = 512
spatial_chunk_dim_zarr_output = 2048
bbox_left = -179.999
bbox_right = 179.999
bbox_top = 81.099
bbox_bottom = -59.999
wy_start = 2015
wy_end = 2024
low_backscatter_threshold = 0.001
min_monthly_acquisitions = 1
max_allowed_days_gap_per_orbit = 30
min_years_for_median_std = 3
extend_search_window_beyond_sdd_days = 16
min_consec_snow_days_for_seasonal_snow = 56
valid_tiles_geojson_path = processing/tile_data/global_tiles_with_seasonal_snow.geojson
tile_results_path = processing/tile_data/tile_results_v9.csv
global_runoff_zarr_store_azure_path = snowmelt/snowmelt_runoff_onset/global_v9.zarr
seasonal_snow_mask_zarr_store_azure_path = snowmelt/snow_cover/global_modis_snow_cover.zarr
seasona

In [3]:
# url = (f"https://data.earthenv.org/mountains/standard/GMBA_Inventory_v2.0_standard_300.zip")
# gmba_gdf = gpd.read_file("zip+" + url)
#gmba_gdf
#gmba_gdf.filter(['MapName','geometry']).explore(column='MapName')
# gmba_gdf.columns
# gmba_gdf.explore(column='Level_03')

In [4]:
# basins_gdf = easysnowdata.hydroclimatology.get_grdc_major_river_basins_of_the_world()
# basins_gdf

In [5]:
# basins_gdf.plot()

In [6]:
# basins_gdf = easysnowdata.hydroclimatology.get_grdc_wmo_basins()
# basins_gdf

In [7]:
# WTU_gdf = gpd.read_file("zip+https://zenodo.org/records/3521933/files/data_ngs_watertowers.zip!/WTU_units/WTU_vector.shp")
# WTU_gdf

In [8]:
#WTU_gdf.explore(column='WTU_NAME')

In [9]:
all_river_basins_ds = xr.open_dataset(f"aggregated_results/river_basins/fcf_lte_50/{config.version}/all_river_basins.nc",decode_times=False)
all_river_basins_ds

<xarray.Dataset> Size: 4GB
Dimensions:                           (river_basin: 1890, elevation: 90,
                                       aspect: 24, statistic: 5, water_year: 10)
Coordinates:
  * river_basin                       (river_basin) int64 15kB 11429 ... 85209
  * elevation                         (elevation) float64 720B 50.0 ... 8.95e+03
  * aspect                            (aspect) float64 192B 0.1309 ... 6.152
  * statistic                         (statistic) <U11 220B 'basin_count' ......
  * water_year                        (water_year) int64 80B 2015 2016 ... 2024
Data variables:
    runoff_onset_median               (river_basin, elevation, aspect, statistic) float64 163MB ...
    runoff_onset_mad                  (river_basin, elevation, aspect, statistic) float64 163MB ...
    runoff_onset                      (river_basin, elevation, aspect, statistic, water_year) float64 2GB ...
    runoff_onset_anomaly              (river_basin, elevation, aspect, statistic, water_year) float64 2GB ...
    runoff_onset_elev_relative        (river_basin, elevation, aspect, statistic) float64 163MB ...
    basin_runoff_onset_median         (statistic, river_basin) float64 76kB ...
    basin_runoff_onset_mad            (statistic, river_basin) float64 76kB ...
    basin_runoff_onset                (statistic, river_basin, water_year) float64 756kB ...
    basin_runoff_onset_anomaly        (statistic, river_basin, water_year) float64 756kB ...
    basin_runoff_onset_elev_relative  (statistic, river_basin) float64 76kB ...
Attributes:
    location:   11429
    unit_type:  PFAF_ID

In [10]:
basin_populations_gdf = gpd.read_file("aggregated_results/river_basins/Hydrobasins_L5_Population_Global.geojson")
basin_populations_gdf

,id,HYBAS_ID,PFAF_ID,total_population,geometry
0,0000000000000000020b,5050558240,56122,9.278136e+03,"POLYGON ((116.24548 -30.69412, 116.24619 -30.6..."
1,0000000000000000020c,5050558410,56123,1.362956e+04,"POLYGON ((116.41668 -32.31458, 116.42048 -32.3..."
2,0000000000000000020d,5050560670,56125,4.885879e+03,"POLYGON ((117.61224 -31.6428, 117.61724 -31.64..."
3,0000000000000000020e,5050560780,56124,5.921051e+03,"POLYGON ((117.62917 -31.91667, 117.63138 -31.9..."
4,0000000000000000020f,5050546000,56126,7.170845e+02,"POLYGON ((118 -30.52917, 118.00221 -30.52973, ..."
...,...,...,...,...,...
4729,00000000000000001279,4050028560,45377,3.725325e+06,"POLYGON ((79.13333 12.35417, 79.1339 12.34362,..."
4730,0000000000000000127a,4050028680,45379,5.320099e+06,"POLYGON ((78.2625 11.60833, 78.27083 11.60833,..."
4731,0000000000000000127b,4050028770,45390,2.347101e+07,"MULTIPOLYGON (((78.72856 9.15261, 78.73194 9.1..."
4732,0000000000000000127c,4050029530,45401,1.589016e+07,"MULTIPOLYGON (((76.37917 9.62083, 76.37978 9.6..."


In [11]:
basin_populations_gdf['total_population'].sum()

print('total population in all basins (billion):', basin_populations_gdf['total_population'].sum()/1e9)

total population in all basins (billion): 7.953138432166601


In [12]:
# f,ax=plt.subplots(figsize=(20,10))
# basin_populations_gdf.plot(ax=ax,column='total_population',legend=True, cmap='viridis',norm=colors.LogNorm(vmin=1, vmax=basin_populations_gdf['total_population'].max()))

In [13]:
#basins_gdf.explore(column='WMOBB_BASIN')

In [14]:
basins_gdf = easysnowdata.hydroclimatology.get_hydroBASINS(level=5)
basins_gdf

Getting geometries for Africa...
Africa takes a bit longer because we have to temporarily save the file due to read issue...
Getting geometries for Arctic...
Getting geometries for Asia...
Getting geometries for Australia...
Getting geometries for Europe...
Getting geometries for Greenland...
Getting geometries for North America...
Getting geometries for South America...
Getting geometries for Siberia...


,HYBAS_ID,NEXT_DOWN,NEXT_SINK,MAIN_BAS,DIST_SINK,DIST_MAIN,SUB_AREA,UP_AREA,PFAF_ID,ENDO,COAST,ORDER,SORT,geometry
0,1050000010,0,1050000010,1050000010,0.0,0.0,45567.0,45567.0,11101,0,1,0,1,"MULTIPOLYGON (((35.17778 24.64583, 35.17279 24..."
1,1050001370,0,1050001370,1050001370,0.0,0.0,11678.4,11678.5,11102,0,0,1,2,"POLYGON ((35.14167 22.58333, 35.15316 22.58271..."
2,1050001380,0,1050001380,1050001380,0.0,0.0,5524.6,5524.6,11103,0,1,0,3,"POLYGON ((35.92083 22.50417, 35.91424 22.50452..."
3,1050001510,0,1050001510,1050001510,0.0,0.0,42400.5,42400.5,11104,0,0,1,4,"POLYGON ((35.24167 21.59167, 35.24202 21.60245..."
4,1050001520,0,1050001520,1050001520,0.0,0.0,16198.5,16198.5,11105,0,1,0,5,"MULTIPOLYGON (((36.63194 22.2875, 36.6286 22.2..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
403,3050024510,3050481930,3050024510,3050001840,0.0,1848.1,13573.8,13573.8,31260,2,0,3,404,"POLYGON ((62.94167 52.67917, 62.9411 52.68138,..."
404,3050024540,3050001840,3050024540,3050001840,0.0,1.0,22286.5,22286.5,31205,2,0,2,405,"POLYGON ((79.77917 51.97917, 79.77973 51.98138..."
405,3050024640,3050679760,3050024640,3050004740,0.0,4239.9,5144.4,5144.4,32280,2,0,3,406,"POLYGON ((105.275 46.975, 105.28554 46.97557, ..."
406,3050024960,3050001840,3050024960,3050001840,0.0,1.0,19166.9,19166.9,31203,2,0,2,407,"POLYGON ((78.225 52.325, 78.22742 52.32535, 78..."


In [15]:
# add 'POPULATION' column to basins_gdf
basins_gdf = basins_gdf.merge(basin_populations_gdf[['HYBAS_ID','total_population']], on='HYBAS_ID', how='left')
basins_gdf = basins_gdf.rename(columns={'total_population': 'POPULATION'})
basins_gdf

,HYBAS_ID,NEXT_DOWN,NEXT_SINK,MAIN_BAS,DIST_SINK,DIST_MAIN,SUB_AREA,UP_AREA,PFAF_ID,ENDO,COAST,ORDER,SORT,geometry,POPULATION
0,1050000010,0,1050000010,1050000010,0.0,0.0,45567.0,45567.0,11101,0,1,0,1,"MULTIPOLYGON (((35.17778 24.64583, 35.17279 24...",454453.709453
1,1050001370,0,1050001370,1050001370,0.0,0.0,11678.4,11678.5,11102,0,0,1,2,"POLYGON ((35.14167 22.58333, 35.15316 22.58271...",7931.771638
2,1050001380,0,1050001380,1050001380,0.0,0.0,5524.6,5524.6,11103,0,1,0,3,"POLYGON ((35.92083 22.50417, 35.91424 22.50452...",6593.424997
3,1050001510,0,1050001510,1050001510,0.0,0.0,42400.5,42400.5,11104,0,0,1,4,"POLYGON ((35.24167 21.59167, 35.24202 21.60245...",117412.544791
4,1050001520,0,1050001520,1050001520,0.0,0.0,16198.5,16198.5,11105,0,1,0,5,"MULTIPOLYGON (((36.63194 22.2875, 36.6286 22.2...",42922.135617
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4729,3050024510,3050481930,3050024510,3050001840,0.0,1848.1,13573.8,13573.8,31260,2,0,3,404,"POLYGON ((62.94167 52.67917, 62.9411 52.68138,...",51916.220892
4730,3050024540,3050001840,3050024540,3050001840,0.0,1.0,22286.5,22286.5,31205,2,0,2,405,"POLYGON ((79.77917 51.97917, 79.77973 51.98138...",104481.186272
4731,3050024640,3050679760,3050024640,3050004740,0.0,4239.9,5144.4,5144.4,32280,2,0,3,406,"POLYGON ((105.275 46.975, 105.28554 46.97557, ...",3848.093983
4732,3050024960,3050001840,3050024960,3050001840,0.0,1.0,19166.9,19166.9,31203,2,0,2,407,"POLYGON ((78.225 52.325, 78.22742 52.32535, 78...",306683.596285


In [16]:
# cluster = coiled.Cluster(idle_timeout="10 minutes",
#                         n_workers=50, #10
#                         worker_memory="32 GB", #32 # 128 and 32 almost worked
#                         worker_cpu=4, # 4,
#                         scheduler_memory="128 GB",
#                         spot_policy="spot",
#                         environ={"GDAL_DISABLE_READDIR_ON_OPEN": "EMPTY_DIR"},
#                         workspace="uwtacolab",
#                         )

# client = cluster.get_client()

In [17]:
# import dask.delayed

# #@dask.delayed
# def process_and_save_river_basin(river_basin_id, parquet_path, filesystem, 
#                                   dem_bin_low=0, dem_bin_high=9000, 
#                                   dem_bin_interval=100, aspect_bin_interval=15):
#     """Process single mountain range and save to netCDF"""
#     #print(f'Processing mountain range {mountain_range_id}...')
    
#     # Setup bins and coordinates
#     dem_bins = np.arange(dem_bin_low, dem_bin_high+dem_bin_interval, dem_bin_interval)
#     aspect_bins = np.arange(0, 360+aspect_bin_interval, aspect_bin_interval)
#     water_years = range(2015, 2025)
#     dem_coords = dem_bins[:-1] + dem_bin_interval/2
#     aspect_coords = aspect_bins[:-1] + aspect_bin_interval/2
#     stat_coords = ["mean", "median", "count"]

#     # Read data
#     cols_needed = ['PFAF_ID', 'dem', 'aspect', 'runoff_onset_median', 'runoff_onset_mad'] + \
#                   [f'runoff_onset_WY{year}' for year in water_years]
    
#     range_ddf = dd.read_parquet(
#         parquet_path,
#         filesystem=filesystem,
#         columns=cols_needed,
#         #storage_options={"anon":       
#         filters=[('PFAF_ID', '==', river_basin_id)]
#     ).repartition(partition_size="100MB").persist()

#     # Create bins and calculate anomalies
#     def bin_and_anomaly(df):
#         df = df.dropna(subset=['dem', 'aspect'])
#         df['dem_bin'] = pd.cut(df['dem'], dem_bins).apply(lambda x: x.left+(dem_bin_interval/2) if pd.notnull(x) else np.nan)
#         df['aspect_bin'] = pd.cut(df['aspect'], aspect_bins).apply(lambda x: x.left+(aspect_bin_interval/2) if pd.notnull(x) else np.nan)
#         df['dem_bin'] = df['dem_bin'].astype('Int64')
#         df['aspect_bin'] = df['aspect_bin'].astype('Float64')
        
#         # Calculate anomalies
#         for year in water_years:
#             col = f'runoff_onset_WY{year}'
#             anom_col = f'runoff_onset_anomaly_WY{year}'
#             df[anom_col] = df[col].where(df[col] > 0) - df['runoff_onset_median']
        
#         return df

#     range_ddf = range_ddf.map_partitions(bin_and_anomaly)
#     range_ddf = range_ddf.dropna(subset=['dem_bin', 'aspect_bin'])
#     range_ddf = range_ddf.persist()

#     # Calculate static statistics
#     #print("Computing static aggregations...")
#     agg_static = range_ddf.groupby(['dem_bin', 'aspect_bin']).agg({
#         'runoff_onset_median': ['mean', 'median', 'count'],
#         'runoff_onset_mad': ['mean', 'median', 'count']
#     }).compute()

#     #print("Computing yearly aggregations...")
#     yearly_aggs = []
#     anomaly_aggs = []
    
#     for year in water_years:
#         year_col = f'runoff_onset_WY{year}'
#         # Filter out -9999 values
#         year_data = range_ddf[range_ddf[year_col] > 0][['dem_bin', 'aspect_bin', year_col]]
        
#         year_agg = year_data.groupby(['dem_bin', 'aspect_bin']).agg({
#             year_col: ['mean', 'median', 'count']
#         }).compute()
        
#         year_agg.columns = ['mean', 'median', 'count']
#         year_agg = year_agg.reset_index()
#         year_agg['water_year'] = year
#         yearly_aggs.append(year_agg)

#         # Process anomaly data that was calculated earlier
#         anom_col = f'runoff_onset_anomaly_WY{year}'
#         anom_data = range_ddf.dropna(subset=[anom_col])[['dem_bin', 'aspect_bin', anom_col]]
        
#         anom_agg = anom_data.groupby(['dem_bin', 'aspect_bin']).agg({
#             anom_col: ['mean', 'median', 'count']
#         }).compute()
        
#         anom_agg.columns = ['mean', 'median', 'count']
#         anom_agg = anom_agg.reset_index()
#         anom_agg['water_year'] = year
#         anomaly_aggs.append(anom_agg)

#     # Convert to pandas DataFrames
#     yearly_aggs = pd.concat(yearly_aggs, ignore_index=True)
#     anomaly_aggs = pd.concat(anomaly_aggs, ignore_index=True)

    
#     # Create dataset
#     ds = xr.Dataset(
#         coords={
#             'elevation': dem_coords,
#             'aspect': aspect_coords,
#             'statistic': stat_coords,
#             'water_year': list(water_years)
#         }
#     )

#     # Initialize arrays
#     shape_static = (len(ds.elevation), len(ds.aspect), len(ds.statistic))
#     shape_yearly = shape_static + (len(ds.water_year),)

#     ds['runoff_onset_median'] = xr.DataArray(np.full(shape_static, np.nan), 
#         dims=('elevation', 'aspect', 'statistic'))
#     ds['runoff_onset_mad'] = xr.DataArray(np.full(shape_static, np.nan), 
#         dims=('elevation', 'aspect', 'statistic'))
#     ds['runoff_onset'] = xr.DataArray(np.full(shape_yearly, np.nan), 
#         dims=('elevation', 'aspect', 'statistic', 'water_year'))
#     ds['runoff_onset_anomaly'] = xr.DataArray(np.full(shape_yearly, np.nan), 
#         dims=('elevation', 'aspect', 'statistic', 'water_year'))

#     # Fill values using pandas indexing
#     for dem in ds.elevation.values:
#         for asp in ds.aspect.values:
#             # Fill static variables
#             static_mask = (agg_static.index.get_level_values('dem_bin') == dem) & \
#                          (agg_static.index.get_level_values('aspect_bin') == asp)
            
#             if static_mask.any():
#                 static_data = agg_static[static_mask]
#                 for stat in ['mean', 'median', 'count']:
#                     ds['runoff_onset_median'].loc[{
#                         'elevation': dem,
#                         'aspect': asp,
#                         'statistic': stat
#                     }] = static_data[('runoff_onset_median', stat)].iloc[0]
                    
#                     ds['runoff_onset_mad'].loc[{
#                         'elevation': dem,
#                         'aspect': asp,
#                         'statistic': stat
#                     }] = static_data[('runoff_onset_mad', stat)].iloc[0]
            
#             # Fill yearly variables
#             year_mask = (yearly_aggs['dem_bin'] == dem) & (yearly_aggs['aspect_bin'] == asp)
#             anom_mask = (anomaly_aggs['dem_bin'] == dem) & (anomaly_aggs['aspect_bin'] == asp)
            
#             for stat in ['mean', 'median', 'count']:
#                 year_data = yearly_aggs[year_mask]
#                 if not year_data.empty:
#                     for _, row in year_data.iterrows():
#                         ds['runoff_onset'].loc[{
#                             'elevation': dem,
#                             'aspect': asp,
#                             'statistic': stat,
#                             'water_year': row['water_year']
#                         }] = row[stat]
                
#                 anom_data = anomaly_aggs[anom_mask]
#                 if not anom_data.empty:
#                     for _, row in anom_data.iterrows():
#                         ds['runoff_onset_anomaly'].loc[{
#                             'elevation': dem,
#                             'aspect': asp,
#                             'statistic': stat,
#                             'water_year': row['water_year']
#                         }] = row[stat]

#     # Add attributes
#     ds.elevation.attrs['units'] = 'meters'
#     ds.aspect.attrs['units'] = 'degrees'
#     ds.runoff_onset.attrs['units'] = 'day of water year'
#     ds.runoff_onset_anomaly.attrs['units'] = 'days'
#     ds.runoff_onset_median.attrs['units'] = 'day of water year'
#     ds.runoff_onset_mad.attrs['units'] = 'days'
    
#     ds.attrs['location'] = river_basin_id

#     return ds.compute()

# @dask.delayed
# def get_river_basins(parquet_path, filesystem):
#     mountain_ranges = dd.read_parquet(parquet_path, 
#                                     filesystem=filesystem,
#                                     columns=['PFAF_ID'],
#                                     )['PFAF_ID'].unique().compute()
#     return mountain_ranges
    
# def process_all_river_basins(parquet_path, output_dir, filesystem, batch_size=30):
#     """Process mountain ranges in batches"""

#     os.makedirs(output_dir, exist_ok=True)

#     # Get all mountain ranges once
#     river_basins = get_river_basins(parquet_path, filesystem).compute()
    
#     # Filter out already processed ranges
#     unprocessed_basins = [
#         rb_id for rb_id in river_basins 
#         if not os.path.exists(f"{output_dir}/river_basin_{rb_id}.nc")
#     ]

#     # Process in batches
#     for i in range(0, len(unprocessed_basins), batch_size):
#         batch = unprocessed_basins[i:i + batch_size]
#         futures = []

#         print(f"Processing batch {i//batch_size + 1} of {(len(unprocessed_basins)-1)//batch_size + 1}")
        
#         # Submit batch of tasks
#         for rb_id in batch:
#             future = client.submit(process_and_save_river_basin, rb_id, parquet_path, filesystem)
#             futures.append(future)

#         # Wait for batch completion and save results
#         for future, result in dask.distributed.as_completed(futures, with_results=True):
#             rb_id = result.attrs['location']
#             output_file = f"{output_dir}/river_basin_{rb_id}.nc"
#             result.to_netcdf(output_file)
#             print(f"Saved to {output_file}")

#         # Optional: restart client between batches to clear memory
#         client.restart()

In [18]:
# client.restart()
# process_all_river_basins(parquet_path="snowmelt/analysis/full_datasets/v5_basins/fcf_lte_50",
#                             output_dir ="aggregated_results/river_basins/v5_basins/fcf_lte_50",
#                             filesystem=config.azure_blob_fs)
# client.restart()

In [19]:
# def merge_river_basin_netcdfs(netcdf_dir):

#     """Merge all river basin netCDFs"""
#     nc_files = sorted(glob.glob(f"{netcdf_dir}/river_basin_*.nc")) # _4* for asia
#     datasets = []
    
#     for nc_file in nc_files:
#         ds = xr.open_dataset(nc_file,decode_times=False)
#         rb_id = int(nc_file.split('_')[-1].split('.')[0])
#         ds = ds.expand_dims({'river_basin': [rb_id]})
#         datasets.append(ds)
    
#     merged_ds = xr.concat(datasets, dim='river_basin')


#     merged_ds['aspect'] = np.deg2rad(merged_ds['aspect'])
#     merged_ds['aspect'].attrs['units'] = 'radians'

#     merged_ds = merged_ds.sel(river_basin=merged_ds.river_basin != -9999)


#     merged_ds = merged_ds.sortby('river_basin')

#     merged_ds['runoff_onset_elev_relative'] = merged_ds['runoff_onset_median'] - merged_ds['runoff_onset_median'].median(dim='aspect')
#     merged_ds['runoff_onset_elev_relative'].loc[{'statistic': 'count'}] = merged_ds['runoff_onset_median'].sel(statistic='count')
    
#     return merged_ds

In [20]:
# netcdf_dir = "aggregated_results/river_basins/v5_basins/fcf_lte_50"
# netcdf_dir = "aggregated_results/river_basins/fcf_lte_50/v9"

# merged_ds = merge_river_basin_netcdfs(netcdf_dir)
# merged_ds

In [21]:
# # create weighted average of variables in each basin, weighted by count of samples in each bin. should create a basin_mean and basin_count variable
# total_count_ds = merged_ds.sel(statistic='count').sum(dim=['elevation','aspect'])

# basin_mean_ds = (merged_ds.sel(statistic='count')*merged_ds.sel(statistic='mean')).sum(dim=['elevation','aspect'])/(total_count_ds)

# weighted_mean_ds = xr.concat([basin_mean_ds, total_count_ds], 
#                            dim=pd.Index(['basin_mean', 'basin_count'], name='statistic'))

# weighted_mean_ds

In [22]:
# first remove all basins from basins_gdf with no data in merged_ds
basins_filtered_gdf = basins_gdf[basins_gdf['PFAF_ID'].isin(all_river_basins_ds.river_basin.values)]

column_dict = {
    'HYBAS_ID': 'first',
    'NEXT_DOWN': 'first',
    'NEXT_SINK': 'first',
    'MAIN_BAS': 'first',
    'DIST_SINK': 'first',
    'DIST_MAIN': 'first',
    'SUB_AREA': np.sum,
    'UP_AREA': 'first',
    'PFAF_ID': 'first',
    'ENDO': 'first',
    'COAST': 'first',
    'ORDER': 'first',
    'SORT': 'first',
    'POPULATION': np.sum,
}

basins_filtered_gdf = basins_filtered_gdf.dissolve(by='PFAF_ID', aggfunc=column_dict).drop(columns=['PFAF_ID']).reset_index()


#basins_filtered_gdf = basins_filtered_gdf[basins_filtered_gdf['PFAF_ID']!=35301] # issue with basin crossing -180/180, causes duplicate basins_gdf[basins_gdf['PFAF_ID'] == 35301].plot()
#weighted_mean_filtered_ds = weighted_mean_ds.where(weighted_mean_ds.river_basin != 35301, drop=True)

basins_filtered_gdf

/home/eric/miniconda3/envs/new_global_snowmelt_runoff_onset/lib/python3.13/site-packages/geopandas/geodataframe.py:2274: FutureWarning: The provided callable <function sum at 0x7fe91b9345e0> is currently using SeriesGroupBy.sum. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "sum" instead.
  aggregated_data = data.groupby(**groupby_kwargs).agg(aggfunc, **kwargs)


,PFAF_ID,geometry,HYBAS_ID,NEXT_DOWN,NEXT_SINK,MAIN_BAS,DIST_SINK,DIST_MAIN,SUB_AREA,UP_AREA,ENDO,COAST,ORDER,SORT,POPULATION
0,11429,"POLYGON ((37.32083 -0.125, 37.3214 -0.12721, 3...",1051118600,1051137860,1050008100,1050008100,558.5,558.5,22823.2,22824.0,0,0,2,40,1.262622e+06
1,11732,"POLYGON ((36.7375 -1.99167, 36.73715 -1.98091,...",1050008570,0,1050008570,1050008570,0.0,0.0,46797.7,46797.7,0,0,1,63,1.005683e+07
2,11734,"POLYGON ((38.58333 -5.12083, 38.58665 -5.12168...",1050009010,0,1050009010,1050009010,0.0,0.0,51382.7,51382.7,0,0,1,65,4.260083e+06
3,12780,"MULTIPOLYGON (((26.51723 -31.46371, 26.51367 -...",1051629430,1051626210,1050015850,1050015850,1240.0,1240.0,98763.6,98763.6,0,0,2,234,2.376479e+06
4,15140,"POLYGON ((-10.24167 27.81667, -10.24259 27.813...",1050028960,0,1050028960,1050028960,0.0,0.0,95886.5,95886.5,0,0,1,554,9.539145e+05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1885,84300,"MULTIPOLYGON (((-100.65 72.175, -100.6258 72.1...",8050028930,0,8050028930,8050028930,0.0,0.0,34998.6,34998.6,0,1,0,199,0.000000e+00
1886,85101,"MULTIPOLYGON (((-88.18229 70.33404, -88.15978 ...",8050032840,0,8050032840,8050032840,0.0,0.0,20406.0,20406.0,0,1,0,202,0.000000e+00
1887,85203,"MULTIPOLYGON (((-77.49166 63.22006, -77.49306 ...",8050039140,0,8050039140,8050039140,0.0,0.0,117062.1,117062.1,0,1,0,213,8.815997e+03
1888,85208,"POLYGON ((-79.45 71.48333, -79.44779 71.4839, ...",8050042910,0,8050042910,8050042910,0.0,0.0,17386.1,17386.3,0,0,1,218,0.000000e+00


In [23]:
# create a new geodataframe with the weighted mean runoff onset date, mad, pfaf id, and geometry. we should assign in some sort of join or merge with the pfaf id to make sure we are referring to the same basins
basins_means_gdf = basins_filtered_gdf[['PFAF_ID', 'SUB_AREA','POPULATION','geometry']].copy()
basins_means_gdf = basins_means_gdf.sort_values(by='PFAF_ID')
basins_means_gdf['runoff_onset_median'] = all_river_basins_ds["basin_runoff_onset_median"].sel(statistic='basin_mean').values
basins_means_gdf['runoff_onset_mad'] = all_river_basins_ds["basin_runoff_onset_mad"].sel(statistic='basin_mean').values

for water_year in all_river_basins_ds['water_year'].values:
    basins_means_gdf[f'runoff_onset_WY{water_year}'] = all_river_basins_ds["basin_runoff_onset"].sel(statistic='basin_mean', water_year=water_year).values
    basins_means_gdf[f'runoff_onset_anomaly_WY{water_year}'] = all_river_basins_ds["basin_runoff_onset_anomaly"].sel(statistic='basin_mean', water_year=water_year).values

basins_means_gdf

,PFAF_ID,SUB_AREA,POPULATION,geometry,runoff_onset_median,runoff_onset_mad,runoff_onset_WY2015,runoff_onset_anomaly_WY2015,runoff_onset_WY2016,runoff_onset_anomaly_WY2016,...,runoff_onset_WY2020,runoff_onset_anomaly_WY2020,runoff_onset_WY2021,runoff_onset_anomaly_WY2021,runoff_onset_WY2022,runoff_onset_anomaly_WY2022,runoff_onset_WY2023,runoff_onset_anomaly_WY2023,runoff_onset_WY2024,runoff_onset_anomaly_WY2024
0,11429,22823.2,1.262622e+06,"POLYGON ((37.32083 -0.125, 37.3214 -0.12721, 3...",213.958333,19.937917,NaN,NaN,NaN,NaN,...,182.285714,-25.809524,282.200000,30.900000,NaN,NaN,NaN,NaN,NaN,NaN
1,11732,46797.7,1.005683e+07,"POLYGON ((36.7375 -1.99167, 36.73715 -1.98091,...",267.635762,43.181300,282.709578,14.630278,269.929907,-8.897196,...,270.343173,1.048893,231.493482,-39.108007,168.065056,-100.874535,323.982318,50.791749,245.468485,-28.385701
2,11734,51382.7,4.260083e+06,"POLYGON ((38.58333 -5.12083, 38.58665 -5.12168...",211.066998,41.212084,196.590354,-23.098780,177.647491,-43.549572,...,220.500792,6.054881,191.874529,-25.157781,152.459200,-62.883733,279.039157,53.888554,238.535918,17.704105
3,12780,98763.6,2.376479e+06,"MULTIPOLYGON (((26.51723 -31.46371, 26.51367 -...",129.694030,7.242388,123.777778,-5.555556,NaN,NaN,...,NaN,NaN,127.531915,-1.531915,131.382609,1.730435,124.206897,-4.362069,NaN,NaN
4,15140,95886.5,9.539145e+05,"POLYGON ((-10.24167 27.81667, -10.24259 27.813...",163.718056,9.747804,178.210320,12.262751,184.531550,14.326782,...,151.268970,-23.038445,168.141256,1.592558,160.171262,-5.338738,166.754225,0.966770,183.195964,-2.967534
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1885,84300,34998.6,0.000000e+00,"MULTIPOLYGON (((-100.65 72.175, -100.6258 72.1...",246.166667,5.080337,NaN,NaN,NaN,NaN,...,251.446691,5.259191,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1886,85101,20406.0,0.000000e+00,"MULTIPOLYGON (((-88.18229 70.33404, -88.15978 ...",239.385074,9.431707,NaN,NaN,NaN,NaN,...,245.126386,5.523878,237.459907,-2.150559,NaN,NaN,NaN,NaN,NaN,NaN
1887,85203,117062.1,8.815997e+03,"MULTIPOLYGON (((-77.49166 63.22006, -77.49306 ...",231.468052,10.248855,NaN,NaN,NaN,NaN,...,229.697594,-2.152775,235.625598,3.802772,NaN,NaN,NaN,NaN,NaN,NaN
1888,85208,17386.1,0.000000e+00,"POLYGON ((-79.45 71.48333, -79.44779 71.4839, ...",232.593552,13.905341,NaN,NaN,NaN,NaN,...,236.973551,4.532485,243.510049,10.876124,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
basins_counts_gdf = basins_filtered_gdf[['PFAF_ID', 'SUB_AREA','POPULATION', 'geometry']].copy()
basins_counts_gdf = basins_counts_gdf.sort_values(by='PFAF_ID')
basins_counts_gdf['runoff_onset_median'] = all_river_basins_ds["basin_runoff_onset_median"].sel(statistic='basin_count').values
basins_counts_gdf['runoff_onset_mad'] = all_river_basins_ds["basin_runoff_onset_mad"].sel(statistic='basin_count').values
for water_year in all_river_basins_ds['water_year'].values:
    basins_counts_gdf[f'runoff_onset_WY{water_year}'] = all_river_basins_ds["basin_runoff_onset"].sel(statistic='basin_count', water_year=water_year).values
    basins_counts_gdf[f'runoff_onset_anomaly_WY{water_year}'] = all_river_basins_ds["basin_runoff_onset_anomaly"].sel(statistic='basin_count', water_year=water_year).values
basins_counts_gdf

,PFAF_ID,SUB_AREA,POPULATION,geometry,runoff_onset_median,runoff_onset_mad,runoff_onset_WY2015,runoff_onset_anomaly_WY2015,runoff_onset_WY2016,runoff_onset_anomaly_WY2016,...,runoff_onset_WY2020,runoff_onset_anomaly_WY2020,runoff_onset_WY2021,runoff_onset_anomaly_WY2021,runoff_onset_WY2022,runoff_onset_anomaly_WY2022,runoff_onset_WY2023,runoff_onset_anomaly_WY2023,runoff_onset_WY2024,runoff_onset_anomaly_WY2024
0,11429,22823.2,1.262622e+06,"POLYGON ((37.32083 -0.125, 37.3214 -0.12721, 3...",24.0,24.0,0.0,0.0,0.0,0.0,...,21.0,21.0,10.0,10.0,0.0,0.0,0.0,0.0,0.0,0.0
1,11732,46797.7,1.005683e+07,"POLYGON ((36.7375 -1.99167, 36.73715 -1.98091,...",1208.0,1208.0,971.0,971.0,214.0,214.0,...,1084.0,1084.0,1074.0,1074.0,1076.0,1076.0,1018.0,1018.0,1063.0,1063.0
2,11734,51382.7,4.260083e+06,"POLYGON ((38.58333 -5.12083, 38.58665 -5.12168...",2015.0,2015.0,1721.0,1721.0,817.0,817.0,...,1895.0,1895.0,1857.0,1857.0,1875.0,1875.0,1660.0,1660.0,1754.0,1754.0
3,12780,98763.6,2.376479e+06,"MULTIPOLYGON (((26.51723 -31.46371, 26.51367 -...",134.0,134.0,18.0,18.0,0.0,0.0,...,0.0,0.0,47.0,47.0,115.0,115.0,58.0,58.0,0.0,0.0
4,15140,95886.5,9.539145e+05,"POLYGON ((-10.24167 27.81667, -10.24259 27.813...",53766.0,53766.0,42074.0,42074.0,18621.0,18621.0,...,13838.0,13838.0,39078.0,39078.0,44651.0,44651.0,42071.0,42071.0,3419.0,3419.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1885,84300,34998.6,0.000000e+00,"MULTIPOLYGON (((-100.65 72.175, -100.6258 72.1...",594.0,594.0,0.0,0.0,0.0,0.0,...,544.0,544.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1886,85101,20406.0,0.000000e+00,"MULTIPOLYGON (((-88.18229 70.33404, -88.15978 ...",276326.0,276326.0,0.0,0.0,0.0,0.0,...,272325.0,272325.0,271509.0,271509.0,0.0,0.0,0.0,0.0,0.0,0.0
1887,85203,117062.1,8.815997e+03,"MULTIPOLYGON (((-77.49166 63.22006, -77.49306 ...",513013.0,513013.0,0.0,0.0,0.0,0.0,...,500999.0,500999.0,501586.0,501586.0,0.0,0.0,0.0,0.0,0.0,0.0
1888,85208,17386.1,0.000000e+00,"POLYGON ((-79.45 71.48333, -79.44779 71.4839, ...",20566.0,20566.0,0.0,0.0,0.0,0.0,...,19963.0,19963.0,20351.0,20351.0,0.0,0.0,0.0,0.0,0.0,0.0


In [25]:
variable_list = ['runoff_onset_median','runoff_onset_mad','runoff_onset_WY2015',
                 'runoff_onset_WY2016','runoff_onset_WY2017','runoff_onset_WY2018',
                 'runoff_onset_WY2019','runoff_onset_WY2020','runoff_onset_WY2021',
                 'runoff_onset_WY2022','runoff_onset_WY2023','runoff_onset_WY2024', 'runoff_onset_anomaly_WY2015',
                 'runoff_onset_anomaly_WY2016','runoff_onset_anomaly_WY2017',
                 'runoff_onset_anomaly_WY2018','runoff_onset_anomaly_WY2019',
                 'runoff_onset_anomaly_WY2020','runoff_onset_anomaly_WY2021',
                 'runoff_onset_anomaly_WY2022','runoff_onset_anomaly_WY2023', 'runoff_onset_anomaly_WY2024']

In [26]:
basins_areas_gdf = basins_counts_gdf.copy()

for variable in variable_list:
    basins_areas_gdf[variable] = basins_counts_gdf[variable]*(((80/1000)*(80/1000)))
    
basins_areas_gdf

,PFAF_ID,SUB_AREA,POPULATION,geometry,runoff_onset_median,runoff_onset_mad,runoff_onset_WY2015,runoff_onset_anomaly_WY2015,runoff_onset_WY2016,runoff_onset_anomaly_WY2016,...,runoff_onset_WY2020,runoff_onset_anomaly_WY2020,runoff_onset_WY2021,runoff_onset_anomaly_WY2021,runoff_onset_WY2022,runoff_onset_anomaly_WY2022,runoff_onset_WY2023,runoff_onset_anomaly_WY2023,runoff_onset_WY2024,runoff_onset_anomaly_WY2024
0,11429,22823.2,1.262622e+06,"POLYGON ((37.32083 -0.125, 37.3214 -0.12721, 3...",0.1536,0.1536,0.0000,0.0000,0.0000,0.0000,...,0.1344,0.1344,0.0640,0.0640,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1,11732,46797.7,1.005683e+07,"POLYGON ((36.7375 -1.99167, 36.73715 -1.98091,...",7.7312,7.7312,6.2144,6.2144,1.3696,1.3696,...,6.9376,6.9376,6.8736,6.8736,6.8864,6.8864,6.5152,6.5152,6.8032,6.8032
2,11734,51382.7,4.260083e+06,"POLYGON ((38.58333 -5.12083, 38.58665 -5.12168...",12.8960,12.8960,11.0144,11.0144,5.2288,5.2288,...,12.1280,12.1280,11.8848,11.8848,12.0000,12.0000,10.6240,10.6240,11.2256,11.2256
3,12780,98763.6,2.376479e+06,"MULTIPOLYGON (((26.51723 -31.46371, 26.51367 -...",0.8576,0.8576,0.1152,0.1152,0.0000,0.0000,...,0.0000,0.0000,0.3008,0.3008,0.7360,0.7360,0.3712,0.3712,0.0000,0.0000
4,15140,95886.5,9.539145e+05,"POLYGON ((-10.24167 27.81667, -10.24259 27.813...",344.1024,344.1024,269.2736,269.2736,119.1744,119.1744,...,88.5632,88.5632,250.0992,250.0992,285.7664,285.7664,269.2544,269.2544,21.8816,21.8816
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1885,84300,34998.6,0.000000e+00,"MULTIPOLYGON (((-100.65 72.175, -100.6258 72.1...",3.8016,3.8016,0.0000,0.0000,0.0000,0.0000,...,3.4816,3.4816,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1886,85101,20406.0,0.000000e+00,"MULTIPOLYGON (((-88.18229 70.33404, -88.15978 ...",1768.4864,1768.4864,0.0000,0.0000,0.0000,0.0000,...,1742.8800,1742.8800,1737.6576,1737.6576,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1887,85203,117062.1,8.815997e+03,"MULTIPOLYGON (((-77.49166 63.22006, -77.49306 ...",3283.2832,3283.2832,0.0000,0.0000,0.0000,0.0000,...,3206.3936,3206.3936,3210.1504,3210.1504,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000
1888,85208,17386.1,0.000000e+00,"POLYGON ((-79.45 71.48333, -79.44779 71.4839, ...",131.6224,131.6224,0.0000,0.0000,0.0000,0.0000,...,127.7632,127.7632,130.2464,130.2464,0.0000,0.0000,0.0000,0.0000,0.0000,0.0000


In [27]:
basins_pct_areas_gdf = basins_areas_gdf.copy()

for variable in variable_list:
    basins_pct_areas_gdf[variable] = (basins_areas_gdf[variable]/basins_counts_gdf['SUB_AREA'])*100
    
basins_pct_areas_gdf

,PFAF_ID,SUB_AREA,POPULATION,geometry,runoff_onset_median,runoff_onset_mad,runoff_onset_WY2015,runoff_onset_anomaly_WY2015,runoff_onset_WY2016,runoff_onset_anomaly_WY2016,...,runoff_onset_WY2020,runoff_onset_anomaly_WY2020,runoff_onset_WY2021,runoff_onset_anomaly_WY2021,runoff_onset_WY2022,runoff_onset_anomaly_WY2022,runoff_onset_WY2023,runoff_onset_anomaly_WY2023,runoff_onset_WY2024,runoff_onset_anomaly_WY2024
0,11429,22823.2,1.262622e+06,"POLYGON ((37.32083 -0.125, 37.3214 -0.12721, 3...",0.000673,0.000673,0.000000,0.000000,0.000000,0.000000,...,0.000589,0.000589,0.000280,0.000280,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,11732,46797.7,1.005683e+07,"POLYGON ((36.7375 -1.99167, 36.73715 -1.98091,...",0.016520,0.016520,0.013279,0.013279,0.002927,0.002927,...,0.014825,0.014825,0.014688,0.014688,0.014715,0.014715,0.013922,0.013922,0.014537,0.014537
2,11734,51382.7,4.260083e+06,"POLYGON ((38.58333 -5.12083, 38.58665 -5.12168...",0.025098,0.025098,0.021436,0.021436,0.010176,0.010176,...,0.023603,0.023603,0.023130,0.023130,0.023354,0.023354,0.020676,0.020676,0.021847,0.021847
3,12780,98763.6,2.376479e+06,"MULTIPOLYGON (((26.51723 -31.46371, 26.51367 -...",0.000868,0.000868,0.000117,0.000117,0.000000,0.000000,...,0.000000,0.000000,0.000305,0.000305,0.000745,0.000745,0.000376,0.000376,0.000000,0.000000
4,15140,95886.5,9.539145e+05,"POLYGON ((-10.24167 27.81667, -10.24259 27.813...",0.358864,0.358864,0.280825,0.280825,0.124287,0.124287,...,0.092363,0.092363,0.260828,0.260828,0.298026,0.298026,0.280805,0.280805,0.022820,0.022820
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1885,84300,34998.6,0.000000e+00,"MULTIPOLYGON (((-100.65 72.175, -100.6258 72.1...",0.010862,0.010862,0.000000,0.000000,0.000000,0.000000,...,0.009948,0.009948,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1886,85101,20406.0,0.000000e+00,"MULTIPOLYGON (((-88.18229 70.33404, -88.15978 ...",8.666502,8.666502,0.000000,0.000000,0.000000,0.000000,...,8.541017,8.541017,8.515425,8.515425,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1887,85203,117062.1,8.815997e+03,"MULTIPOLYGON (((-77.49166 63.22006, -77.49306 ...",2.804736,2.804736,0.000000,0.000000,0.000000,0.000000,...,2.739054,2.739054,2.742263,2.742263,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1888,85208,17386.1,0.000000e+00,"POLYGON ((-79.45 71.48333, -79.44779 71.4839, ...",0.757055,0.757055,0.000000,0.000000,0.000000,0.000000,...,0.734858,0.734858,0.749141,0.749141,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000


In [28]:
# f,axs = plt.subplots(2,1,figsize=(10,8), subplot_kw={'projection': ccrs.Robinson()}, dpi=300, layout='constrained')

# basins_means_gdf.plot(ax=axs[0],column='runoff_onset_median', cmap='viridis', legend=True, legend_kwds={'label': "Median runoff onset date [DOWY]"},transform=ccrs.PlateCarree(),vmin=110,vmax=250)
# basins_means_gdf.plot(ax=axs[1],column='runoff_onset_mad', cmap='Reds', legend=True, legend_kwds={'label': "Runoff onset MAD [days]"},transform=ccrs.PlateCarree(),vmin=0,vmax=30)

# for ax in axs:
#     gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=[-180, -120, -60, 0, 60, 120, 180], ylocs=[-60, -40, -20, 0, 20, 40, 60, 80], linestyle='--', linewidth=0.5)
#     gl.top_labels=False
#     gl.bottom_labels=True
#     gl.right_labels=False
#     gl.left_labels=True

#     ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

#     ax.add_feature(cfeature.LAND, facecolor='lightgrey')
#     ax.add_feature(cfeature.OCEAN, facecolor='powderblue') # 'lightcyan'
#     ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')

In [29]:
# f,axs = plt.subplots(3,2,figsize=(30,20), subplot_kw={'projection': ccrs.PlateCarree()}, dpi=300, layout='constrained',sharex=True, sharey=True)

# basins_counts_gdf.plot(ax=axs[0,0],column='runoff_onset_median', cmap='viridis', legend=True, legend_kwds={'label': "Median runoff onset pixel count per basin","shrink":0.5},transform=ccrs.PlateCarree(),norm=colors.LogNorm())
# basins_counts_gdf.plot(ax=axs[0,1],column='runoff_onset_mad', cmap='Reds', legend=True, legend_kwds={'label': "Runoff onset MAD pixel count per basin","shrink":0.5},transform=ccrs.PlateCarree(),norm=colors.LogNorm())

# basins_areas_gdf.plot(ax=axs[1,0],column='runoff_onset_median', cmap='viridis', legend=True, legend_kwds={'label': "Median runoff onset pixel area per basin [km^2]","shrink":0.5},transform=ccrs.PlateCarree())
# basins_areas_gdf.plot(ax=axs[1,1],column='runoff_onset_mad', cmap='Reds', legend=True, legend_kwds={'label': "Runoff onset MAD pixel area per basin [km^2]","shrink":0.5},transform=ccrs.PlateCarree())

# basins_pct_areas_gdf.plot(ax=axs[2,0],column='runoff_onset_median', cmap='viridis', legend=True, legend_kwds={'label': "Median runoff onset pixel pct area per basin [%]","shrink":0.5},transform=ccrs.PlateCarree())
# basins_pct_areas_gdf.plot(ax=axs[2,1],column='runoff_onset_mad', cmap='Reds', legend=True, legend_kwds={'label': "Runoff onset MAD pixel pct area per basin [%]","shrink":0.5},transform=ccrs.PlateCarree())

# for ax in axs.flat:
#     gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=[-180, -120, -60, 0, 60, 120, 180], ylocs=[-60, -40, -20, 0, 20, 40, 60, 80], linestyle='--', linewidth=0.5)
#     gl.top_labels=False
#     gl.bottom_labels=True
#     gl.right_labels=False
#     gl.left_labels=True

#     ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

#     ax.add_feature(cfeature.LAND, facecolor='lightgrey')
#     ax.add_feature(cfeature.OCEAN, facecolor='powderblue') # 'lightcyan'
#     ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')


In [ ]:
f,ax=plt.subplots(1,1,figsize=(15,10), subplot_kw={'projection': ccrs.AlbersEqualArea(central_longitude=0.0, central_latitude=45.0)}, dpi=300, layout='constrained')
basins_means_aea_gdf.plot(ax=ax,column='POPULATION', cmap='plasma', legend=True, legend_kwds={'label': "Basin Population"},transform=cartopy_aea_crs, norm=colors.LogNorm(vmin=100, vmax=pop_max),edgecolor='black', linewidth=0.2)

In [ ]:
hillshade_da = rxr.open_rasterio('../analysis/figures/methods/global_hillshade_robinson.tif', masked=True, chunks='auto').squeeze().coarsen(x=10,y=10, boundary='trim').mean().compute()
hillshade_da

In [ ]:
basin_means_filtered_gdf = basins_means_gdf.copy()
# mask out basins with less than 5% of area in the basin
for variable in variable_list:
    basin_means_filtered_gdf[variable] = basin_means_filtered_gdf[variable].where(basins_pct_areas_gdf['runoff_onset_median'] > 5, np.nan)
    for water_year in all_river_basins_ds['water_year'].values:
        basin_means_filtered_gdf[f'runoff_onset_WY{water_year}'] = basin_means_filtered_gdf[f'runoff_onset_WY{water_year}'].where(basins_pct_areas_gdf[f'runoff_onset_WY{water_year}'] > 1, np.nan)
        basin_means_filtered_gdf[f'runoff_onset_anomaly_WY{water_year}'] = basin_means_filtered_gdf[f'runoff_onset_anomaly_WY{water_year}'].where(basins_pct_areas_gdf[f'runoff_onset_anomaly_WY{water_year}'] > 1, np.nan)

In [ ]:
basins_means_robinson_gdf = basin_means_filtered_gdf.to_crs("ESRI:54030")

In [ ]:
f,axs = plt.subplots(2,1,figsize=(10,8), subplot_kw={'projection': ccrs.Robinson()}, dpi=300, layout='constrained')

basins_means_robinson_gdf.plot(ax=axs[0],column='runoff_onset_median', cmap='viridis', legend=True, legend_kwds={'label': "Median runoff onset date [DOWY]"},transform=ccrs.Robinson(),vmin=110,vmax=250)
basins_means_robinson_gdf.plot(ax=axs[1],column='runoff_onset_mad', cmap='Reds', legend=True, legend_kwds={'label': "Runoff onset MAD [days]"},transform=ccrs.Robinson(),vmin=0,vmax=30)

for ax in axs:
    
    hillshade_da.plot.imshow(ax=ax,cmap='gray', transform=ccrs.Robinson(), zorder=0,add_colorbar=False)
    gl = ax.gridlines(draw_labels=False, dms=True, x_inline=False, y_inline=False, xlocs=[-180, -120, -60, 0, 60, 120, 180], ylocs=[-90, -60, -30, 0, 30, 60, 90], linestyle='--', linewidth=0.5)
    # gl.top_labels=False
    # gl.bottom_labels=True
    # gl.right_labels=False
    # gl.left_labels=True
    ax.set_title("")
    #ax.set_global()

    #ax.set_extent([-180, 180, -63, 90], crs=ccrs.PlateCarree())

    # ax.add_feature(cfeature.LAND, facecolor='lightgrey')
    # ax.add_feature(cfeature.OCEAN, facecolor='powderblue') # 'lightcyan'
    # ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')
    
axs[0].set_title("10-year Median Snowmelt Runoff Onset by River Basin")
axs[1].set_title("10-year Median Absolute Deviation of Snowmelt Runoff Onset by River Basin")

f.savefig(f'figures/river_basins/global_median_runoff_onset_and_mad.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
all_river_basins_ds['basin_runoff_onset_anomaly'].sel(water_year=2022, statistic='basin_mean').idxmin()

In [ ]:
basins_means_gdf.loc[basins_means_gdf['PFAF_ID']==45255].plot()

In [ ]:
hillshade_da

In [ ]:
# f,ax=plt.subplots(nrows=5,ncols=2,figsize=(5,6),subplot_kw={'projection': ccrs.Robinson()},layout='constrained', sharex=True, sharey=True) # ccrs.Robinson()

# for water_year, ax in zip(all_river_basins_ds.water_year.values, ax.flat):
#     basins_means_robinson_gdf.plot(ax=ax,column=f'runoff_onset_anomaly_WY{water_year}',cmap='RdBu',vmin=-30,vmax=30,transform=ccrs.Robinson())
#     ax.set_global()

In [ ]:
f,ax=plt.subplots(nrows=5,ncols=2,figsize=(10,12),subplot_kw={'projection': ccrs.Robinson()},layout='constrained', sharex=True, sharey=True) # ccrs.Robinson()

for water_year, ax in zip(all_river_basins_ds.water_year.values, ax.flat):
    basins_means_robinson_gdf.plot(ax=ax,column=f'runoff_onset_anomaly_WY{water_year}',cmap='RdBu',vmin=-30,vmax=30,transform=ccrs.Robinson())
    hillshade_da.plot.imshow(ax=ax,cmap='gray', transform=ccrs.Robinson(), zorder=0,add_colorbar=False)
    ax.set_title(f'WY{water_year}')

    #ax.set_global()
    #gl = ax.gridlines(draw_labels=False, dms=True, x_inline=False, y_inline=False, xlocs=[-180, -120, -60, 0, 60, 120, 180], ylocs=[-90, -60, -30, 0, 30, 60, 90], linestyle='--', linewidth=0.5)
    # gl.top_labels=False
    # gl.bottom_labels=True
    # gl.right_labels=False
    # gl.left_labels=True

    ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

    # ax.add_feature(cfeature.LAND, facecolor='lightgrey')
    # ax.add_feature(cfeature.OCEAN, facecolor='powderblue') # 'lightcyan'
    # ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')

f.savefig(f'figures/river_basins/global_runoff_onset_anomaly_by_basin.png', bbox_inches='tight', dpi=300)
#f.suptitle('Runoff Onset Anomaly by River Basin')
# f.tight_layout()

In [ ]:
f,axes=plt.subplots(nrows=2,ncols=5,figsize=(15,6.2),subplot_kw={'projection': ccrs.Robinson()},layout='constrained', sharex=True, sharey=True) # ccrs.Robinson()

for water_year, ax in zip(all_river_basins_ds.water_year.values, axes.flat):
    basins_means_robinson_gdf.plot(ax=ax,column=f'runoff_onset_anomaly_WY{water_year}',cmap='RdBu',vmin=-30,vmax=30,transform=ccrs.Robinson())
    ax.set_title(f'WY{water_year}')
    gl = ax.gridlines(draw_labels=False, dms=True, x_inline=False, y_inline=False, linestyle='--', linewidth=0.5)
    #gl.top_labels=False
    # gl.bottom_labels=True
    #gl.right_labels=False
    # gl.left_labels=True

    # turn on left labels for WY2015 and WY2020 only, turn on bottom labels for WY 2020-WY2024
    # if water_year == 2015 or water_year == 2020:
    #     gl.left_labels = True
    # else:
    #     gl.left_labels = False
    # if water_year in [2020, 2021, 2022, 2023, 2024]:
    #     gl.bottom_labels = True
    # else:
    #     gl.bottom_labels = False


# Add a single colorbar spanning both rows
import matplotlib as mpl
norm = mpl.colors.Normalize(vmin=-30, vmax=30)
sm = plt.cm.ScalarMappable(cmap='RdBu', norm=norm)
sm.set_array([])
cbar = f.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.8, aspect=20, pad=0.02, extend='both')
cbar.set_label('Snowmelt runoff onset anomaly [days]')


In [ ]:
region = 'hma'
#region = 'wus'

if region == 'hma':
    cartopy_aea_crs = ccrs.AlbersEqualArea(central_latitude=32.5, central_longitude=90)
if region == 'wus':
    cartopy_aea_crs = ccrs.AlbersEqualArea(central_latitude=39, central_longitude=-120)
    
basins_means_aea_gdf = basin_means_filtered_gdf.to_crs(cartopy_aea_crs)

In [ ]:
if region == 'hma':
    hillshade_regional_da = hillshade_da.rio.clip_box(minx=55, miny=10, maxx=115, maxy=55, crs="EPSG:4326").rio.reproject(cartopy_aea_crs).coarsen(x=2,y=2, boundary='trim').mean()
if region == 'wus':
    hillshade_regional_da = hillshade_da.rio.clip_box(minx=-170, miny=25, maxx=-100, maxy=80, crs="EPSG:4326").rio.reproject(cartopy_aea_crs).coarsen(x=2,y=2, boundary='trim').mean()
hillshade_regional_da

In [ ]:
#hillshade_regional_da.plot.imshow(cmap='gray',vmin=0,vmax=255)

In [ ]:
# basins_means_wus_gdf = basins_means_aea_gdf.cx[-0.2E7:0.2E7, -0.35E7:0.35E7]
# basins_means_wus_gdf
basins_means_roi_gdf = basins_means_aea_gdf.cx[-0.2E7:0.2E7, -0.35E7:0.35E7]
basins_means_roi_gdf

In [ ]:
#basins_means_wus_gdf.plot()
basins_means_roi_gdf.plot()

In [ ]:
for i,basin in enumerate(basins_means_roi_gdf['PFAF_ID'].values):
    print(f"Processing basin {basin}... {i+1}/{len(basins_means_roi_gdf['PFAF_ID'].values)}")
    try:

        basin_pct_precip_as_snow_da = pct_precip_as_snow_da.rio.clip(basins_means_roi_gdf[basins_means_roi_gdf['PFAF_ID'] == basin].geometry.values, crs=basins_means_roi_gdf.crs)
        basins_means_roi_gdf.loc[basins_means_roi_gdf['PFAF_ID'] == basin, 'pct_precip_as_snow'] = basin_pct_precip_as_snow_da.where(lambda x: x>0).mean().values
    except Exception as e:
        print(f"Error processing basin {basin}: {e}")
        basins_means_roi_gdf.loc[basins_means_roi_gdf['PFAF_ID'] == basin, 'pct_precip_as_snow'] = np.nan

In [ ]:
# filter basins_means_aea_gdf to only include High mountain asia region, use lat lon bounds of 60E to 115E and 10N to 55N
# basins_means_hma_gdf = basins_means_aea_gdf.cx[-0.2E7:0.2E7, -0.2E7:0.2E7]
# basins_means_hma_gdf

# for i,basin in enumerate(basins_means_hma_gdf['PFAF_ID'].values):
#     print(f"Processing basin {basin}... {i+1}/{len(basins_means_hma_gdf['PFAF_ID'].values)}")
#     try:

#         basin_pct_precip_as_snow_da = pct_precip_as_snow_da.rio.clip(basins_means_hma_gdf[basins_means_hma_gdf['PFAF_ID'] == basin].geometry.values, crs=basins_means_hma_gdf.crs)

#         basins_means_hma_gdf.loc[basins_means_hma_gdf['PFAF_ID'] == basin, 'pct_precip_as_snow'] = basin_pct_precip_as_snow_da.where(lambda x: x>0).mean().values
#     except Exception as e:
#         print(f"Error processing basin {basin}: {e}")
#         basins_means_hma_gdf.loc[basins_means_hma_gdf['PFAF_ID'] == basin, 'pct_precip_as_snow'] = np.nan

In [ ]:
f,axes=plt.subplots(nrows=2,ncols=5,figsize=(15,5.5),subplot_kw={'projection': cartopy_aea_crs},layout='constrained', sharex=True, sharey=True) # ccrs.Robinson()

for water_year, ax in zip(all_river_basins_ds.water_year.values, axes.flat):
    basins_means_aea_gdf.plot(ax=ax,column=f'runoff_onset_anomaly_WY{water_year}',cmap='RdBu',vmin=-30,vmax=30,transform=cartopy_aea_crs,edgecolor='black', linewidth=0.2)
    hillshade_regional_da.plot.imshow(ax=ax,cmap='gray', transform=cartopy_aea_crs, zorder=0,add_colorbar=False,vmin=0,vmax=255)

    ax.set_title(f'WY{water_year}')
    # gl = ax.gridlines(draw_labels=False, dms=True, x_inline=False, y_inline=False, linestyle='--', linewidth=0.5)
    # #gl.top_labels=False
    # # gl.bottom_labels=True
    # #gl.right_labels=False
    # # gl.left_labels=True

    # # turn on left labels for WY2015 and WY2020 only, turn on bottom labels for WY 2020-WY2024
    # if water_year == 2015 or water_year == 2020:
    #     gl.left_labels = True
    # else:
    #     gl.left_labels = False
    # if water_year in [2020, 2021, 2022, 2023, 2024]:
    #     gl.bottom_labels = True
    # else:
    #     gl.bottom_labels = False
        

    if region == 'hma':
        ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())
    if region == 'wus':
        ax.set_extent([ -155, -105, 35, 73], crs=ccrs.PlateCarree())
    #ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())
    #ax.set_extent([ -155, -105, 35, 73], crs=ccrs.PlateCarree())
    

    # ax.add_feature(cfeature.LAND, facecolor='lightgrey')
    # ax.add_feature(cfeature.OCEAN, facecolor='powderblue') # 'lightcyan'
    # ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')

# Add a single colorbar spanning both rows
import matplotlib as mpl
norm = mpl.colors.Normalize(vmin=-30, vmax=30)
sm = plt.cm.ScalarMappable(cmap='RdBu', norm=norm)
sm.set_array([])
cbar = f.colorbar(sm, ax=axes.ravel().tolist(), shrink=0.8, aspect=20, pad=0.02, extend='both')
cbar.set_label('Snowmelt runoff onset anomaly [days]')

#f.savefig(f'figures/river_basins/hma_runoff_onset_anomaly_by_basin.png', bbox_inches='tight', dpi=300)
f.savefig(f'figures/river_basins/{region}_runoff_onset_anomaly_by_basin.png', bbox_inches='tight', dpi=300)
#f.suptitle('Runoff Onset Anomaly by River Basin')
# f.tight_layout()

In [ ]:
# now create a two panel figure with the mean and mad runoff onset dates for the region
f,axs = plt.subplots(1,2,figsize=(12,4), subplot_kw={'projection': cartopy_aea_crs}, dpi=300, layout='constrained')
basins_means_aea_gdf.plot(ax=axs[0],column='runoff_onset_median', cmap='viridis', legend=True, legend_kwds={'label': "Median runoff onset date [DOWY]"},transform=cartopy_aea_crs,vmin=110,vmax=250,edgecolor='black', linewidth=0.2)
basins_means_aea_gdf.plot(ax=axs[1],column='runoff_onset_mad', cmap='Reds', legend=True, legend_kwds={'label': "Runoff onset MAD [days]"},transform=cartopy_aea_crs,vmin=0,vmax=30,edgecolor='black', linewidth=0.2)
for ax in axs:
    
    hillshade_regional_da.plot.imshow(ax=ax,cmap='gray', transform=cartopy_aea_crs, zorder=0,add_colorbar=False,vmin=0,vmax=255)
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=[60, 70, 80, 90, 100, 110], ylocs=[20, 30, 40, 50], linestyle='--', linewidth=0.5)
    gl.top_labels=False
    gl.bottom_labels=True
    gl.right_labels=False
    gl.left_labels=True

    ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())

axs[0].set_title("10-year Median Snowmelt Runoff Onset by River Basin")
axs[1].set_title("10-year Median Absolute Deviation of Snowmelt Runoff Onset by River Basin")

In [ ]:
basins_means_roi_gdf.columns


In [ ]:
import matplotlib.colors as colors

In [ ]:
# now create a two panel figure with basin population and pct precip falling as snow

if region == 'hma':
    pop_max = 1E8
if region == 'wus':
    pop_max = 1E7
    
f,axs = plt.subplots(1,2,figsize=(10,4), subplot_kw={'projection': cartopy_aea_crs}, dpi=300, layout='constrained')
basins_means_aea_gdf.plot(ax=axs[0],column='POPULATION', cmap='plasma', legend=True, legend_kwds={'label': "Basin Population"},transform=cartopy_aea_crs, norm=colors.LogNorm(vmin=100, vmax=pop_max),edgecolor='black', linewidth=0.2)
basins_means_roi_gdf.plot(ax=axs[1],column='pct_precip_as_snow', cmap='Blues', legend=True, legend_kwds={'label': "%"},transform=cartopy_aea_crs,edgecolor='black', linewidth=0.2,vmin=0,vmax=60)
for ax in axs:
    
    hillshade_regional_da.plot.imshow(ax=ax,cmap='gray', transform=cartopy_aea_crs, zorder=0,add_colorbar=False,vmin=0,vmax=255)
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=[60, 70, 80, 90, 100, 110], ylocs=[20, 30, 40, 50], linestyle='--', linewidth=0.5)
    gl.top_labels=False
    gl.bottom_labels=True
    gl.right_labels=False
    gl.left_labels=True

    if region == 'hma':
        ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())
    if region == 'wus':
        ax.set_extent([ -155, -105, 35, 73], crs=ccrs.PlateCarree())
    
axs[0].set_title("Basin Population")
axs[1].set_title("Basin Percentage of Precipitation Falling as Snow")

f.savefig(f'figures/river_basins/{region}_basin_population_and_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
import matplotlib.colors as colors

In [ ]:
basins_means_aea_gdf.plot(column='runoff_onset_median')

In [ ]:
important_rivers_gdf = gpd.read_file('geometries/majorrivers_0_0/MajorRivers.shp')
important_rivers_gdf = important_rivers_gdf.to_crs(basins_means_aea_gdf.crs).cx[-0.2E7:0.2E7, -0.35E7:0.2E7]
# include Indus, Ganges, Brahmaputra, Yangtze, Mekong, Amu Darya, Syr Darya, Huang He
important_rivers_gdf = important_rivers_gdf[important_rivers_gdf['NAME'].isin(['Indus', 'Ganges', 'Brahmaputra', 'Yangtze', 'Mekong', 'Amu Darya', 'Syr Darya', 'Huang He'])]
important_rivers_gdf

In [ ]:
#important_rivers_gdf.explore(column='NAME')

In [ ]:
important_rivers_gdf.plot(column='NAME',legend=True)

In [ ]:
f,ax=plt.subplots(figsize=(10,5),subplot_kw={'projection': cartopy_aea_crs}, dpi=300, layout='constrained')
hillshade_regional_da.plot.imshow(ax=ax,cmap='gray', transform=cartopy_aea_crs, zorder=0,add_colorbar=False,vmin=0,vmax=255)
important_rivers_gdf.plot(ax=ax, column='NAME', linewidth=1, zorder=1, legend=True)
# now label the rivers clo
for idx, row in important_rivers_gdf.iterrows():
    centroid = row['geometry'].centroid
    ax.text(centroid.x, centroid.y, row['NAME'], fontsize=8, zorder=2)
ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())
#ax.legend()

In [ ]:
# now make a 4x4 figure of median runoff onset, mad, population, and pct precip as snow for HMA region
f,axs = plt.subplots(2,2,figsize=(10,7), subplot_kw={'projection': cartopy_aea_crs}, dpi=300, layout='constrained')
basins_means_aea_gdf.plot(ax=axs[0,0],column='runoff_onset_median', cmap='viridis', legend=True, legend_kwds={'label': "Median runoff onset date [DOWY]"},transform=cartopy_aea_crs,vmin=110,vmax=250,edgecolor='black', linewidth=0.2, alpha=0.8)
basins_means_aea_gdf.plot(ax=axs[0,1],column='runoff_onset_mad', cmap='Reds', legend=True, legend_kwds={'label': "Runoff onset MAD [days]"},transform=cartopy_aea_crs,vmin=0,vmax=30,edgecolor='black', linewidth=0.2,alpha=0.8)
basins_means_aea_gdf.plot(ax=axs[1,0],column='POPULATION', cmap='plasma', legend=True, legend_kwds={'label': "Basin Population"},transform=cartopy_aea_crs, norm=colors.LogNorm(vmin=100, vmax=1E8),edgecolor='black', linewidth=0.2, alpha=0.8)
basins_means_roi_gdf.plot(ax=axs[1,1],column='pct_precip_as_snow', cmap='Blues', legend=True, legend_kwds={'label': "%"},transform=cartopy_aea_crs,edgecolor='black', linewidth=0.2,vmin=0,vmax=60, alpha=0.8)

for ax in axs.flat:
    
    hillshade_regional_da.plot.imshow(ax=ax,cmap='gray', transform=cartopy_aea_crs, zorder=0,add_colorbar=False,vmin=0,vmax=255)
    # gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=[60, 70, 80, 90, 100, 110], ylocs=[20, 30, 40, 50], linestyle='--', linewidth=0.5)
    # gl.top_labels=False
    # gl.bottom_labels=True
    # gl.right_labels=False
    # gl.left_labels=True
    
    important_rivers_gdf.plot(ax=ax, color='black', linewidth=1, zorder=2)

    if region == 'hma':
        ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())
    if region == 'wus':
        ax.set_extent([ -155, -105, 35, 73], crs=ccrs.PlateCarree())
    #ax.set_extent([65, 108, 20, 50], crs=ccrs.PlateCarree())
    #ax.set_extent([ -155, -105, 35, 73], crs=ccrs.PlateCarree())
    
axs[0,0].set_title("10-year Median Snowmelt Runoff Onset")
axs[0,1].set_title("10-year Median Absolute Deviation of Snowmelt Runoff Onset")
axs[1,0].set_title("Basin Population")
axs[1,1].set_title("Basin Percentage of Precipitation Falling as Snow")

f.savefig(f'figures/river_basins/{region}_basin_runoff_onset_population_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# # now create a scatter plot, with dot size representing basin population, x axis as pct precip as snow, and y axis as runoff onset median, colored by runoff onset mad
# f,ax = plt.subplots(figsize=(8,6), dpi=300, layout='constrained')
# sc = ax.scatter(basins_means_hma_gdf['pct_precip_as_snow'], basins_means_hma_gdf['runoff_onset_median'], 
#                 s=basins_means_hma_gdf['POPULATION']/1000, 
#                 c=basins_means_hma_gdf['runoff_onset_mad'], cmap='Reds', alpha=0.7, edgecolors='k', linewidth=0.5)
# cbar = plt.colorbar(sc, ax=ax)
# cbar.set_label('Runoff Onset MAD [days]')
# ax.set_xlabel('Percentage of Precipitation Falling as Snow [%]')
# ax.set_ylabel('Median Snowmelt Runoff Onset Date [DOWY]')
# ax.set_title('HMA River Basins: Runoff Onset vs. % Precip as Snow\n(Dot size represents Basin Population)')
# f.savefig(f'figures/river_basins/hma_basin_runoff_onset_vs_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# now create a scatter plot, with dot size representing basin population, x axis as pct precip as snow, and y axis as runoff onset median, colored by runoff onset mad
f,ax = plt.subplots(figsize=(8,6), dpi=300, layout='constrained')

# parameters: base markersize (points) for 1e4, radius multiplier per order of magnitude = 1.5
base_markersize = 8.0
radius_multiplier = 1.5

# clip populations and compute order of magnitude offset from 1e4
pop = basins_means_hma_gdf['POPULATION'].clip(lower=1)
log_pop = np.log10(pop)
magnitude_offset = log_pop - 4.0  # 0 for 1e4, 1 for 1e5, 2 for 1e6, etc.

# Radius (markersize in matplotlib) scales by 1.5 per order of magnitude
marker_radii = base_markersize * (radius_multiplier ** magnitude_offset)

# scatter() 's' parameter expects area in points^2, so we square the radii
s_sizes = marker_radii ** 2

sc = ax.scatter(
    basins_means_hma_gdf['pct_precip_as_snow'],
    basins_means_hma_gdf['runoff_onset_median'],
    s=s_sizes,
    c=basins_means_hma_gdf['runoff_onset_mad'],
    cmap='Reds',
    alpha=0.8,
    edgecolors='k',
    linewidth=0.5,
    vmax=30,
    vmin=5
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Runoff Onset MAD [days]')

# Create legend with 5 population categories (1e4 to 1e8)
legend_populations = [1e4, 1e5, 1e6, 1e7, 1e8]
legend_magnitude_offsets = np.log10(np.array(legend_populations)) - 4.0
legend_marker_radii = base_markersize * (radius_multiplier ** legend_magnitude_offsets)
legend_labels = [f'{int(p/1e6)}M' if p >= 1e6 else f'{int(p/1e3)}K' for p in legend_populations]

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='gray', 
           markersize=radius,  # Use radius directly for legend markers
           label=label,
           markeredgecolor='k', markeredgewidth=0.5)
    for radius, label in zip(legend_marker_radii, legend_labels)
]

ax.legend(handles=legend_elements, title='Basin Population', loc='lower right', framealpha=0.9)

ax.set_xlabel('Percentage of Precipitation Falling as Snow [%]')
ax.set_ylabel('Median Snowmelt Runoff Onset Date [DOWY]')
ax.set_title('HMA River Basins: Runoff Onset vs. % Precip as Snow\n(Dot size represents Basin Population)')
f.savefig(f'figures/river_basins/hma_basin_runoff_onset_vs_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# now create a scatter plot, with dot size representing basin population, x axis as pct precip as snow, and y axis as runoff onset median, colored by runoff onset mad
f,ax = plt.subplots(figsize=(8,6), dpi=300, layout='constrained')

# parameters: base markersize (points) for 1e4, radius multiplier per order of magnitude = 1.5
base_markersize = 8.0
radius_multiplier = 1.5

# clip populations and compute order of magnitude offset from 1e4
pop = basins_means_hma_gdf['POPULATION'].clip(lower=1)
log_pop = np.log10(pop)
magnitude_offset = log_pop - 4.0  # 0 for 1e4, 1 for 1e5, 2 for 1e6, etc.

# Radius (markersize in matplotlib) scales by 1.5 per order of magnitude
marker_radii = base_markersize * (radius_multiplier ** magnitude_offset)

# scatter() 's' parameter expects area in points^2, so we square the radii
s_sizes = marker_radii ** 2

sc = ax.scatter(
    basins_means_hma_gdf['pct_precip_as_snow'],
    basins_means_hma_gdf['runoff_onset_median'],
    s=s_sizes,
    c=basins_means_hma_gdf['runoff_onset_mad'],
    cmap='Reds',
    alpha=0.8,
    edgecolors='k',
    linewidth=0.5,
    vmax=30,
    vmin=5
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Runoff Onset MAD [days]')

# Create legend with 5 population categories
# Adjust to start from smaller value if needed
min_pop = basins_means_hma_gdf['POPULATION'].min()
legend_populations = [1e3, 1e4, 1e5, 1e6, 1e7]  # Start from 1K instead of 10K
legend_magnitude_offsets = np.log10(np.array(legend_populations)) - 4.0
legend_marker_radii = base_markersize * (radius_multiplier ** legend_magnitude_offsets)
legend_labels = [f'{int(p/1e6)}M' if p >= 1e6 else f'{int(p/1e3)}K' for p in legend_populations]

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='gray', 
           markersize=radius,
           label=label,
           markeredgecolor='k', markeredgewidth=0.5)
    for radius, label in zip(legend_marker_radii, legend_labels)
]

# Use labelspacing to prevent overlap
ax.legend(handles=legend_elements, title='Basin Population', 
         loc='lower right', framealpha=0.9, 
         labelspacing=1.5,  # Increase spacing between legend items
         borderpad=1.0)      # Add padding around legend

ax.set_xlabel('Percentage of Precipitation Falling as Snow [%]')
ax.set_ylabel('Median Snowmelt Runoff Onset Date [DOWY]')
ax.set_title('HMA River Basins: Runoff Onset vs. % Precip as Snow\n(Dot size represents Basin Population)')
f.savefig(f'figures/river_basins/hma_basin_runoff_onset_vs_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# now create a scatter plot, with dot size representing basin population, x axis as pct precip as snow, and y axis as runoff onset median, colored by runoff onset mad
f,ax = plt.subplots(figsize=(8,6), dpi=300, layout='constrained')

# parameters: base markersize (points) for 1e4, radius multiplier per order of magnitude = 1.5
base_markersize = 8.0
radius_multiplier = 1.5

# clip populations and compute order of magnitude offset from 1e4
pop = basins_means_hma_gdf['POPULATION'].clip(lower=1)
log_pop = np.log10(pop)
magnitude_offset = log_pop - 4.0  # 0 for 1e4, 1 for 1e5, 2 for 1e6, etc.

# Radius (markersize in matplotlib) scales by 1.5 per order of magnitude
marker_radii = base_markersize * (radius_multiplier ** magnitude_offset)

# scatter() 's' parameter expects area in points^2, so we square the radii
s_sizes = marker_radii ** 2

sc = ax.scatter(
    basins_means_hma_gdf['pct_precip_as_snow'],
    basins_means_hma_gdf['runoff_onset_median'],
    s=s_sizes,
    c=basins_means_hma_gdf['runoff_onset_mad'],
    cmap='Reds',
    alpha=0.8,
    edgecolors='k',
    linewidth=0.5,
    vmax=30,
    vmin=5
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Runoff Onset MAD [days]')

# Add month bands to y-axis
# Calculate DOWY for first day of each month in water year 2022
month_colors = ['#e6f2ff', '#cce5ff', '#b3d9ff', '#99ccff', '#80bfff', '#66b3ff', 
                '#4da6ff', '#3399ff', '#1a8cff', '#0080ff', '#0073e6', '#0066cc']
months = ['Oct', 'Nov', 'Dec', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep']

# Water year starts Oct 1
month_starts = []
for month in range(10, 13):  # Oct, Nov, Dec of 2021
    dowy = easysnowdata.utils.datetime_to_DOWY(f"2021-{month:02d}-01")
    month_starts.append(dowy)
for month in range(1, 10):  # Jan-Sep of 2022
    dowy = easysnowdata.utils.datetime_to_DOWY(f"2022-{month:02d}-01")
    month_starts.append(dowy)

# Add the end of water year
month_starts.append(366)

# Draw month bands
for i in range(len(months)):
    ax.axhspan(month_starts[i], month_starts[i+1], 
               facecolor=month_colors[i], alpha=0.15, zorder=0)

# Create legend with 6 population categories
legend_populations = [1e3, 1e4, 1e5, 1e6, 1e7, 1e8]  # 1K to 100M
legend_magnitude_offsets = np.log10(np.array(legend_populations)) - 4.0
legend_marker_radii = base_markersize * (radius_multiplier ** legend_magnitude_offsets)
legend_labels = [f'{int(p/1e6)}M' if p >= 1e6 else f'{int(p/1e3)}K' for p in legend_populations]

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='gray', 
           markersize=radius,
           label=label,
           markeredgecolor='k', markeredgewidth=0.5)
    for radius, label in zip(legend_marker_radii, legend_labels)
]

# Use labelspacing to prevent overlap
ax.legend(handles=legend_elements, title='Basin Population', 
         loc='lower right', framealpha=0.9, 
         labelspacing=1.8,  # Increased for 6 items
         borderpad=1.0)

ax.set_xlabel('Percentage of Precipitation Falling as Snow [%]')
ax.set_ylabel('Median Snowmelt Runoff Onset Date [DOWY]')
ax.set_title('HMA River Basins: Runoff Onset vs. % Precip as Snow\n(Dot size represents Basin Population)')

# Add secondary y-axis with month labels
ax2 = ax.twinx()
ax2.set_ylim(ax.get_ylim())
ax2.set_yticks([month_starts[i] + (month_starts[i+1] - month_starts[i])/2 for i in range(len(months))])
ax2.set_yticklabels(months, fontsize=8)
ax2.set_ylabel('')

ax.set_ylim(100, 270)
ax2.set_ylim(100, 270)

f.savefig(f'figures/river_basins/hma_basin_runoff_onset_vs_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# now create a scatter plot, with dot size representing basin population, x axis as pct precip as snow, and y axis as runoff onset median, colored by runoff onset mad
f,ax = plt.subplots(figsize=(8,6), dpi=300, layout='constrained')

# parameters: base markersize (points) for 1e4, radius multiplier per order of magnitude = 1.5
base_markersize = 8.0
radius_multiplier = 1.5

# clip populations and compute order of magnitude offset from 1e4
pop = basins_means_hma_gdf['POPULATION'].clip(lower=1)
log_pop = np.log10(pop)
magnitude_offset = log_pop - 4.0  # 0 for 1e4, 1 for 1e5, 2 for 1e6, etc.

# Radius (markersize in matplotlib) scales by 1.5 per order of magnitude
marker_radii = base_markersize * (radius_multiplier ** magnitude_offset)

# scatter() 's' parameter expects area in points^2, so we square the radii
s_sizes = marker_radii ** 2

sc = ax.scatter(
    basins_means_hma_gdf['pct_precip_as_snow'],
    basins_means_hma_gdf['runoff_onset_median'],
    s=s_sizes,
    c=basins_means_hma_gdf['runoff_onset_mad'],
    cmap='Reds',
    alpha=0.8,
    edgecolors='k',
    linewidth=0.5,
    vmax=30,
    vmin=5
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('10-yr runoff onset MAD [days]')

# Add month dividers to y-axis
# Calculate DOWY for first day of each month in water year 2022
months = ['October', 'November', 'December', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September']

# Water year starts Oct 1
month_starts = []
for month in range(10, 13):  # Oct, Nov, Dec of 2021
    dowy = easysnowdata.utils.datetime_to_DOWY(f"2021-{month:02d}-01")
    month_starts.append(dowy)
for month in range(1, 10):  # Jan-Sep of 2022
    dowy = easysnowdata.utils.datetime_to_DOWY(f"2022-{month:02d}-01")
    month_starts.append(dowy)

# Add the end of water year
month_starts.append(366)

# Draw horizontal dashed lines at month boundaries and add month labels
for i in range(len(months)):
    # Draw dashed line at the start of each month
    # only draw for Jan-Jun, otherwise skip to avoid clutter

    if months[i] in ['January', 'February', 'March', 'April', 'May', 'June']:

        ax.axhline(y=month_starts[i], color='black', linestyle='--', linewidth=0.5, alpha=0.5, zorder=1)
        
        # Add month label in the middle of each month's range, rotated vertically
        mid_point = (month_starts[i] + month_starts[i+1]) / 2
        ax.text(6, mid_point, months[i], rotation=90, va='center', ha='center', fontsize=9, color='black')

# Create legend with 6 population categories
legend_populations = [1e3, 1e4, 1e5, 1e6, 1e7, 1e8]  # 1K to 100M
legend_magnitude_offsets = np.log10(np.array(legend_populations)) - 4.0
legend_marker_radii = base_markersize * (radius_multiplier ** legend_magnitude_offsets)
legend_labels = [f'{int(p/1e6)}M' if p >= 1e6 else f'{int(p/1e3)}K' for p in legend_populations]

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='gray', 
           markersize=radius,
           label=label,
           markeredgecolor='k', markeredgewidth=0.5)
    for radius, label in zip(legend_marker_radii, legend_labels)
]

# Use labelspacing to prevent overlap
ax.legend(handles=legend_elements, 
         title='Basin population', 
         loc='lower center', 
         bbox_to_anchor=(0.58, 0.003),
         ncol=6,  # 6 columns for 6 items
         framealpha=1,
         columnspacing=1.3,
         #borderpad=1.5,
         handleheight=3.5,
         # make the legend taller to fit the large marker sizes
         handletextpad=1)


ax.set_xlabel('Percentage of precipitation falling as snow [%]')
ax.set_ylabel('Median snowmelt runoff onset date [DOWY]')
ax.set_title('HMA river basins: interannual variability')

ax.set_ylim(100, 265)
ax.set_xlim(5, 75)  # Extend x-limit slightly left to accommodate month labels

f.savefig(f'figures/river_basins/hma_basin_runoff_onset_vs_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
# now create a scatter plot, with dot size representing basin population, x axis as pct precip as snow, and y axis as runoff onset median, colored by runoff onset mad
f,ax = plt.subplots(figsize=(8,6), dpi=300, layout='constrained')

# parameters: base markersize (points) for 1e4, radius multiplier per order of magnitude = 1.5
base_markersize = 8.0
radius_multiplier = 1.5

# clip populations and compute order of magnitude offset from 1e4
pop = basins_means_wus_gdf['POPULATION'].clip(lower=1)
log_pop = np.log10(pop)
magnitude_offset = log_pop - 4.0  # 0 for 1e4, 1 for 1e5, 2 for 1e6, etc.

# Radius (markersize in matplotlib) scales by 1.5 per order of magnitude
marker_radii = base_markersize * (radius_multiplier ** magnitude_offset)

# scatter() 's' parameter expects area in points^2, so we square the radii
s_sizes = marker_radii ** 2

sc = ax.scatter(
    basins_means_wus_gdf['pct_precip_as_snow'],
    basins_means_wus_gdf['runoff_onset_median'],
    s=s_sizes,
    c=basins_means_wus_gdf['runoff_onset_mad'],
    cmap='Reds',
    alpha=0.8,
    edgecolors='k',
    linewidth=0.5,
    vmax=30,
    vmin=5
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('10-yr runoff onset MAD [days]')

# Add month dividers to y-axis
# Calculate DOWY for first day of each month in water year 2022
months = ['October', 'November', 'December', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September']

# Water year starts Oct 1
month_starts = []
for month in range(10, 13):  # Oct, Nov, Dec of 2021
    dowy = easysnowdata.utils.datetime_to_DOWY(f"2021-{month:02d}-01")
    month_starts.append(dowy)
for month in range(1, 10):  # Jan-Sep of 2022
    dowy = easysnowdata.utils.datetime_to_DOWY(f"2022-{month:02d}-01")
    month_starts.append(dowy)

# Add the end of water year
month_starts.append(366)

# Draw horizontal dashed lines at month boundaries and add month labels
for i in range(len(months)):
    # Draw dashed line at the start of each month
    # only draw for Jan-Jun, otherwise skip to avoid clutter

    if months[i] in ['January', 'February', 'March', 'April', 'May', 'June']:

        ax.axhline(y=month_starts[i], color='black', linestyle='--', linewidth=0.5, alpha=0.5, zorder=1)
        
        # Add month label in the middle of each month's range, rotated vertically
        mid_point = (month_starts[i] + month_starts[i+1]) / 2
        ax.text(6, mid_point, months[i], rotation=90, va='center', ha='center', fontsize=9, color='black')

# Create legend with 6 population categories
legend_populations = [1e3, 1e4, 1e5, 1e6, 1e7]  # 1K to 100M
legend_magnitude_offsets = np.log10(np.array(legend_populations)) - 4.0
legend_marker_radii = base_markersize * (radius_multiplier ** legend_magnitude_offsets)
legend_labels = [f'{int(p/1e6)}M' if p >= 1e6 else f'{int(p/1e3)}K' for p in legend_populations]

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='gray', 
           markersize=radius,
           label=label,
           markeredgecolor='k', markeredgewidth=0.5)
    for radius, label in zip(legend_marker_radii, legend_labels)
]

# Use labelspacing to prevent overlap
ax.legend(handles=legend_elements, 
         title='Basin population', 
         loc='lower center', 
         bbox_to_anchor=(0.65, 0.003),
         ncol=6,  # 6 columns for 6 items
         framealpha=1,
         columnspacing=1.3,
         #borderpad=1.5,
         handleheight=3.5,
         # make the legend taller to fit the large marker sizes
         handletextpad=1)


ax.set_xlabel('Percentage of precipitation falling as snow [%]')
ax.set_ylabel('Median snowmelt runoff onset date [DOWY]')
ax.set_title('Western North America river basins: interannual variability')

ax.set_ylim(100, 265)
ax.set_xlim(5, 75)  # Extend x-limit slightly left to accommodate month labels

f.savefig(f'figures/river_basins/wus_basin_runoff_onset_vs_pct_precip_as_snow.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
anomaly_cols = [f'runoff_onset_anomaly_WY{year}' for year in all_river_basins_ds.water_year.values]

# Get column name with largest absolute value
basins_means_wus_gdf['largest_anomaly_idxmax'] = basins_means_wus_gdf[anomaly_cols].abs().idxmax(axis=1, skipna=True)

# Create a mask for rows where at least one anomaly value exists
has_data = basins_means_wus_gdf[anomaly_cols].notna().any(axis=1)

# Get the actual value (with sign) from that column, only for rows with data
basins_means_wus_gdf['largest_anomaly_value'] = np.nan
basins_means_wus_gdf.loc[has_data, 'largest_anomaly_value'] = basins_means_wus_gdf[has_data].apply(
    lambda row: row[row['largest_anomaly_idxmax']], 
    axis=1
)

In [ ]:
# now create a scatter plot, with dot size representing basin population, x axis as pct precip as snow, and y axis as runoff onset median, colored by runoff onset mad
f,ax = plt.subplots(figsize=(8,6), dpi=300, layout='constrained')

# parameters: base markersize (points) for 1e4, radius multiplier per order of magnitude = 1.5
base_markersize = 8.0
radius_multiplier = 1.5

# clip populations and compute order of magnitude offset from 1e4
pop = basins_means_wus_gdf['POPULATION'].clip(lower=1)
log_pop = np.log10(pop)
magnitude_offset = log_pop - 4.0  # 0 for 1e4, 1 for 1e5, 2 for 1e6, etc.

# Radius (markersize in matplotlib) scales by 1.5 per order of magnitude
marker_radii = base_markersize * (radius_multiplier ** magnitude_offset)

# scatter() 's' parameter expects area in points^2, so we square the radii
s_sizes = marker_radii ** 2

sc = ax.scatter(
    basins_means_wus_gdf['pct_precip_as_snow'],
    basins_means_wus_gdf['runoff_onset_median'],
    s=s_sizes,
    c=basins_means_wus_gdf['largest_anomaly_value'],
    cmap='RdBu',
    alpha=0.8,
    edgecolors='k',
    linewidth=0.5,
    vmax=30,
    vmin=-30
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Largest runoff onset anomaly WY2015-2024 [days]')

# Add month dividers to y-axis
# Calculate DOWY for first day of each month in water year 2022
months = ['October', 'November', 'December', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September']

# Water year starts Oct 1
month_starts = []
for month in range(10, 13):  # Oct, Nov, Dec of 2021
    dowy = easysnowdata.utils.datetime_to_DOWY(f"2021-{month:02d}-01")
    month_starts.append(dowy)
for month in range(1, 10):  # Jan-Sep of 2022
    dowy = easysnowdata.utils.datetime_to_DOWY(f"2022-{month:02d}-01")
    month_starts.append(dowy)

# Add the end of water year
month_starts.append(366)

# Draw horizontal dashed lines at month boundaries and add month labels
for i in range(len(months)):
    # Draw dashed line at the start of each month
    # only draw for Jan-Jun, otherwise skip to avoid clutter

    if months[i] in ['January', 'February', 'March', 'April', 'May', 'June']:

        ax.axhline(y=month_starts[i], color='black', linestyle='--', linewidth=0.5, alpha=0.5, zorder=1)
        
        # Add month label in the middle of each month's range, rotated vertically
        mid_point = (month_starts[i] + month_starts[i+1]) / 2
        ax.text(6, mid_point, months[i], rotation=90, va='center', ha='center', fontsize=9, color='black')

# Create legend with 6 population categories
legend_populations = [1e3, 1e4, 1e5, 1e6, 1e7]  # 1K to 100M
legend_magnitude_offsets = np.log10(np.array(legend_populations)) - 4.0
legend_marker_radii = base_markersize * (radius_multiplier ** legend_magnitude_offsets)
legend_labels = [f'{int(p/1e6)}M' if p >= 1e6 else f'{int(p/1e3)}K' for p in legend_populations]

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='gray', 
           markersize=radius,
           label=label,
           markeredgecolor='k', markeredgewidth=0.5)
    for radius, label in zip(legend_marker_radii, legend_labels)
]

# Use labelspacing to prevent overlap
ax.legend(handles=legend_elements, 
         title='Basin population', 
         loc='lower center', 
         bbox_to_anchor=(0.65, 0.003),
         ncol=6,  # 6 columns for 6 items
         framealpha=1,
         columnspacing=1.3,
         #borderpad=1.5,
         handleheight=3.5,
         # make the legend taller to fit the large marker sizes
         handletextpad=1)


ax.set_xlabel('Percentage of precipitation falling as snow [%]')
ax.set_ylabel('Median snowmelt runoff onset date [DOWY]')
ax.set_title('Western North America river basins: largest runoff onset anomaly WY2015-2024')

ax.set_ylim(100, 265)
ax.set_xlim(5, 75)  # Extend x-limit slightly left to accommodate month labels

f.savefig(f'figures/river_basins/wus_basin_runoff_onset_vs_pct_precip_as_snow_largest_anom.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
anomaly_cols = [f'runoff_onset_anomaly_WY{year}' for year in all_river_basins_ds.water_year.values]

# Get column name with largest absolute value
basins_means_hma_gdf['largest_anomaly_idxmax'] = basins_means_hma_gdf[anomaly_cols].abs().idxmax(axis=1, skipna=True)

# Create a mask for rows where at least one anomaly value exists
has_data = basins_means_hma_gdf[anomaly_cols].notna().any(axis=1)

# Get the actual value (with sign) from that column, only for rows with data
basins_means_hma_gdf['largest_anomaly_value'] = np.nan
basins_means_hma_gdf.loc[has_data, 'largest_anomaly_value'] = basins_means_hma_gdf[has_data].apply(
    lambda row: row[row['largest_anomaly_idxmax']], 
    axis=1
)

In [ ]:
# now create a scatter plot, with dot size representing basin population, x axis as pct precip as snow, and y axis as runoff onset median, colored by runoff onset mad
f,ax = plt.subplots(figsize=(8,6), dpi=300, layout='constrained')

# parameters: base markersize (points) for 1e4, radius multiplier per order of magnitude = 1.5
base_markersize = 8.0
radius_multiplier = 1.5

# clip populations and compute order of magnitude offset from 1e4
pop = basins_means_hma_gdf['POPULATION'].clip(lower=1)
log_pop = np.log10(pop)
magnitude_offset = log_pop - 4.0  # 0 for 1e4, 1 for 1e5, 2 for 1e6, etc.

# Radius (markersize in matplotlib) scales by 1.5 per order of magnitude
marker_radii = base_markersize * (radius_multiplier ** magnitude_offset)

# scatter() 's' parameter expects area in points^2, so we square the radii
s_sizes = marker_radii ** 2

sc = ax.scatter(
    basins_means_hma_gdf['pct_precip_as_snow'],
    basins_means_hma_gdf['runoff_onset_median'],
    s=s_sizes,
    c=basins_means_hma_gdf['largest_anomaly_value'],
    cmap='RdBu',
    alpha=0.8,
    edgecolors='k',
    linewidth=0.5,
    vmin=-30,
    vmax=30
)

cbar = plt.colorbar(sc, ax=ax)
cbar.set_label('Largest runoff onset anomaly WY2015-2024 [days]')

# Add month dividers to y-axis
# Calculate DOWY for first day of each month in water year 2022
months = ['October', 'November', 'December', 'January', 'February', 'March', 'April', 'May', 'June', 'July', 'August', 'September']

# Water year starts Oct 1
month_starts = []
for month in range(10, 13):  # Oct, Nov, Dec of 2021
    dowy = easysnowdata.utils.datetime_to_DOWY(f"2021-{month:02d}-01")
    month_starts.append(dowy)
for month in range(1, 10):  # Jan-Sep of 2022
    dowy = easysnowdata.utils.datetime_to_DOWY(f"2022-{month:02d}-01")
    month_starts.append(dowy)

# Add the end of water year
month_starts.append(366)

# Draw horizontal dashed lines at month boundaries and add month labels
for i in range(len(months)):
    # Draw dashed line at the start of each month
    # only draw for Jan-Jun, otherwise skip to avoid clutter

    if months[i] in ['January', 'February', 'March', 'April', 'May', 'June']:

        ax.axhline(y=month_starts[i], color='black', linestyle='--', linewidth=0.5, alpha=0.5, zorder=1)
        
        # Add month label in the middle of each month's range, rotated vertically
        mid_point = (month_starts[i] + month_starts[i+1]) / 2
        ax.text(6, mid_point, months[i], rotation=90, va='center', ha='center', fontsize=9, color='black')

# Create legend with 6 population categories
legend_populations = [1e3, 1e4, 1e5, 1e6, 1e7, 1e8]  # 1K to 100M
legend_magnitude_offsets = np.log10(np.array(legend_populations)) - 4.0
legend_marker_radii = base_markersize * (radius_multiplier ** legend_magnitude_offsets)
legend_labels = [f'{int(p/1e6)}M' if p >= 1e6 else f'{int(p/1e3)}K' for p in legend_populations]

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w',
           markerfacecolor='gray', 
           markersize=radius,
           label=label,
           markeredgecolor='k', markeredgewidth=0.5)
    for radius, label in zip(legend_marker_radii, legend_labels)
]

# Use labelspacing to prevent overlap
ax.legend(handles=legend_elements, 
         title='Basin population', 
         loc='lower center', 
         bbox_to_anchor=(0.57, 0.003),
         ncol=6,  # 6 columns for 6 items
         framealpha=1,
         columnspacing=1.3,
         #borderpad=1.5,
         handleheight=3.5,
         # make the legend taller to fit the large marker sizes
         handletextpad=1)


ax.set_xlabel('Percentage of precipitation falling as snow [%]')
ax.set_ylabel('Median snowmelt runoff onset date [DOWY]')
ax.set_title('HMA river basins: largest runoff onset anomaly WY2015-2024')

ax.set_ylim(100, 265)
ax.set_xlim(5, 75)  # Extend x-limit slightly left to accommodate month labels

f.savefig(f'figures/river_basins/hma_basin_runoff_onset_vs_pct_precip_as_snow_largest_anom.png', bbox_inches='tight', dpi=300, transparent=True)

In [ ]:
easysnowdata.utils.datetime_to_DOWY("2022-01-01")

In [ ]:
# bring in ERA5-Land SWE
# crop to basins
# wscatterplot SWE vs MAD per basin? order my SWE amount at peak?

In [ ]:
ee.Initialize(project="egagli-data-access", 
              opt_url='https://earthengine-highvolume.googleapis.com')

In [ ]:
hemisphere = "NH" # "NH" or "SH"
water_year = 2015
water_years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

datasets = []

for water_year in water_years:

    april1_date = f"{water_year}-04-01"
    nh_bbox_input = (-180,0,180,90)

    era5_land_swe_nh_da = easysnowdata.hydroclimatology.get_era5(
                                                            version="ERA5_LAND",
                                                            bbox_input=nh_bbox_input,
                                                            cadence="DAILY",
                                                            start_date=april1_date,
                                                            end_date=april1_date,
                                                            initialize_ee=False,
                                                            variables="snow_depth_water_equivalent")["snow_depth_water_equivalent"]

    era5_land_swe_nh_da = era5_land_swe_nh_da.assign_coords(
        water_year=era5_land_swe_nh_da.time.dt.year
    ).swap_dims(
        {"time": "water_year"}
    ).drop_vars("time")


    october1_date = f"{water_year}-10-01"
    sh_bbox_input = (-180,-90,180,0)

    era5_land_swe_sh_da = easysnowdata.hydroclimatology.get_era5(
                                                            version="ERA5_LAND",
                                                            bbox_input=sh_bbox_input,
                                                            cadence="DAILY",
                                                            start_date=october1_date,
                                                            end_date=october1_date,
                                                            initialize_ee=False,
                                                            variables="snow_depth_water_equivalent")["snow_depth_water_equivalent"]

    era5_land_swe_sh_da = era5_land_swe_sh_da.assign_coords(
        water_year=era5_land_swe_sh_da.time.dt.year
    ).swap_dims(
        {"time": "water_year"}
    ).drop_vars("time")

    datasets.append(xr.merge([era5_land_swe_nh_da, era5_land_swe_sh_da])["snow_depth_water_equivalent"].squeeze())

era5_land_swe_da = xr.concat(datasets, dim='water_year')
era5_land_swe_da = era5_land_swe_da.where(lambda x: x >= 0)
era5_land_swe_da

In [ ]:
era5_land_swe_da.plot.imshow(col='water_year', col_wrap=5, vmin=0, vmax=1, cmap='Blues', add_colorbar=True, cbar_kwargs={'label': 'SWE [m]'})

In [ ]:
era5_land_swe_median_da = era5_land_swe_da.median(dim='water_year')
era5_land_swe_median_da

In [ ]:
era5_land_swe_anomaly_da = era5_land_swe_da - era5_land_swe_median_da
era5_land_swe_anomaly_da.plot.imshow(col='water_year', col_wrap=2, aspect=2, add_colorbar=True, vmin=-0.4, vmax=0.4, cmap='RdBu', cbar_kwargs={'label': 'SWE anomaly [m]'})

In [ ]:
era5_land_swe_pct_norm_da = 100*(era5_land_swe_da/era5_land_swe_median_da)
era5_land_swe_pct_norm_da.plot.imshow(col='water_year', col_wrap=2, aspect=2, add_colorbar=True, vmin=50, vmax=150, cmap='RdBu', cbar_kwargs={'label': 'SWE pct norm [%]'})

In [ ]:
# basin = 21302
# f,ax=plt.subplots(figsize=(10,10)) # ccrs.Robinson()
# era5_land_swe_pct_norm_da.isel(water_year=0).rio.clip(basins_means_gdf[basins_means_gdf['PFAF_ID'] == basin].geometry.values, crs=basins_means_gdf.crs).plot.imshow(ax=ax, cmap='RdBu', add_colorbar=True, cbar_kwargs={'label': 'SWE pct norm [%]'})
# #era5_land_swe_median_da.rio.clip(basins_means_gdf[basins_means_gdf['PFAF_ID'] == basin].geometry.values, crs=basins_means_gdf.crs).plot.imshow(ax=ax, add_colorbar=True, cbar_kwargs={'label': 'SWE pct norm [%]'})

# basins_means_gdf[basins_means_gdf['PFAF_ID'] == basin].plot(ax=ax, facecolor='none', edgecolor='black', linewidth=1)

In [ ]:
R = 6.371e6

dϕ = np.deg2rad(0.1)
dλ = np.deg2rad(0.1)

dlat = R * dϕ * xr.ones_like(era5_land_swe_da['longitude'])
dlon = R * dλ * np.cos(np.deg2rad(era5_land_swe_da['latitude']))
dlon.name = "dlon"
dlat.name = "dlat"

cell_area_m_da = dlon * dlat
cell_area_m_da=cell_area_m_da.rio.write_crs(era5_land_swe_da.rio.crs)
cell_area_km_da = cell_area_m_da / 1e6

surface_area = cell_area_km_da.sum()
print(f"Total surface area of the dataset: {surface_area.values:,.0f} km²")


cell_area_km_da

In [ ]:
# Initialize with high volume endpoint
ee.Initialize(project="egagli-data-access", 
              opt_url='https://earthengine-highvolume.googleapis.com')

# Create yearly sums ImageCollection as before
variables = ["snowfall_sum", "total_precipitation_sum"]
image_collection = ee.ImageCollection("ECMWF/ERA5_LAND/MONTHLY_AGGR").select(variables)

start_year = 1950
end_year = 2024

def process_year(year):
    start_date = ee.Date.fromYMD(year, 1, 1)
    end_date = ee.Date.fromYMD(year, 12, 31)
    
    # Filter images for this year
    yearly_collection = image_collection.filterDate(start_date, end_date)
    
    # Sum monthly values to get annual totals
    yearly_sum = yearly_collection.sum()
    
    # Add year as a property
    return yearly_sum.set('year', year)

# Generate list of years and map the function over each year
years = range(start_year, end_year + 1)
yearly_sums = ee.ImageCollection(list(map(lambda y: process_year(y), years)))


mean_image = yearly_sums.mean()

snowfall = mean_image.select('snowfall_sum')
total_precip = mean_image.select('total_precipitation_sum')

valid_precip = total_precip.gt(0)  # Mask where precip > 0
pct_snow = snowfall.divide(total_precip).multiply(100).updateMask(valid_precip)
pct_snow = pct_snow.reproject(crs=image_collection.first().projection())
pct_snow = pct_snow.rename('percent_snow')


projection = pct_snow.projection()

pct_precip_as_snow_da = xr.open_dataset(
    ee.ImageCollection(pct_snow),
    engine='ee',
    projection=projection,

)['percent_snow'].squeeze().compute()

pct_precip_as_snow_da

In [ ]:
pct_precip_as_snow_da = (pct_precip_as_snow_da
        .transpose('lat', 'lon')
        .rename({'lat': 'latitude', 'lon': 'longitude'})
        .rio.set_spatial_dims(x_dim='longitude', y_dim='latitude'))
pct_precip_as_snow_da

In [ ]:
f,ax=plt.subplots(figsize=(12,6), subplot_kw={'projection': ccrs.Robinson()}, dpi=300, layout='constrained')

# Plot the data
# Define the color scheme similar to the one in the image
# Define the color scheme
colors = ['#7F00FF', '#5000FF', '#2E00FF', '#0000FF', '#003AFF', '#0075FF', 
          '#00AFFF', '#00C8C8', '#00E282', '#37FF00', '#69FF00', '#A0FF00',
          '#D7FF00', '#FFDC00', '#FFA500', '#FF6B00', '#FF3200', '#FF0000', '#FF0080']

# Create custom colormap
cmap = matplotlib.colors.LinearSegmentedColormap.from_list('custom_cmap', colors)
cmap.set_under('white')  # This sets values below vmin to white

# Set the levels for the color scale
levels = np.arange(0, 105, 5)  # 0, 5, 10, 15, ..., 95


plot = pct_precip_as_snow_da.plot(ax=ax, transform=ccrs.PlateCarree(),
               #cmap=cmap, levels=levels, 
               cmap='Purples',
               cbar_kwargs={'label': 'Percentage [%]'},
               add_colorbar=True)

# gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
# gl.top_labels = False
# gl.right_labels = False

ax.coastlines()

ax.set_title('Percentage of annual water-equivalent precipitation that falls as snow')


In [ ]:
water_years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

datasets = []

for water_year in water_years:

    march1_date = f"{water_year}-03-01"
    may31_date = f"{water_year}-05-31"
    nh_bbox_input = (-180,0,180,90)

    era5_land_spring_temp_nh_da = easysnowdata.hydroclimatology.get_era5(
                                                            version="ERA5_LAND",
                                                            bbox_input=nh_bbox_input,
                                                            cadence="MONTHLY",
                                                            start_date=march1_date,
                                                            end_date=may31_date,
                                                            initialize_ee=False,
                                                            variables="temperature_2m")["temperature_2m"]

    era5_land_spring_temp_nh_da = era5_land_spring_temp_nh_da.mean(dim='time').expand_dims({'water_year':[era5_land_spring_temp_nh_da.time.dt.year[0]]})


    september1_date = f"{water_year}-09-01"
    november30_date = f"{water_year}-11-30"
    sh_bbox_input = (-180,-90,180,0)

    era5_land_spring_temp_sh_da = easysnowdata.hydroclimatology.get_era5(
                                                            version="ERA5_LAND",
                                                            bbox_input=sh_bbox_input,
                                                            cadence="MONTHLY",
                                                            start_date=september1_date,
                                                            end_date=november30_date,
                                                            initialize_ee=False,
                                                            variables="temperature_2m")["temperature_2m"]

    era5_land_spring_temp_sh_da = era5_land_spring_temp_sh_da.mean(dim='time').expand_dims({'water_year':[era5_land_spring_temp_sh_da.time.dt.year[0]]})

    datasets.append(xr.merge([era5_land_spring_temp_nh_da, era5_land_spring_temp_sh_da])["temperature_2m"].squeeze())

era5_land_spring_temp_da = xr.concat(datasets, dim='water_year')
era5_land_spring_temp_da = era5_land_spring_temp_da.where(lambda x: x >= 0)
era5_land_spring_temp_da = era5_land_spring_temp_da - 273.15
era5_land_spring_temp_10yr_median_da = era5_land_spring_temp_da.median(dim='water_year')
era5_land_spring_temp_10yr_anomaly_da = era5_land_spring_temp_da - era5_land_spring_temp_10yr_median_da

era5_land_spring_temp_10yr_anomaly_da

In [ ]:
era5_land_spring_temp_10yr_anomaly_da.plot.imshow(col='water_year', col_wrap=5, cmap='RdBu_r', vmin=-5, vmax=5)

In [ ]:
basins_means_gdf

In [ ]:
# for each basin, add SWE median, SWE for each water year, and SWE anomaly for each water year from the era5_land_swe_da cropped to the basin

for i,basin in enumerate(basins_means_gdf['PFAF_ID'].values):
    print(f"Processing basin {basin}... {i+1}/{len(basins_means_gdf['PFAF_ID'].values)}")
    try:
        basin_swe_da = era5_land_swe_da.rio.clip(basins_means_gdf[basins_means_gdf['PFAF_ID'] == basin].geometry.values, crs=basins_means_gdf.crs)
        basin_swe_median_da = basin_swe_da.median(dim='water_year')
        basin_swe_anomaly_da = basin_swe_da - basin_swe_median_da
        basin_swe_pct_norm_da = 100*(basin_swe_da/basin_swe_median_da)

        basin_pct_precip_as_snow_da = pct_precip_as_snow_da.rio.clip(basins_means_gdf[basins_means_gdf['PFAF_ID'] == basin].geometry.values, crs=basins_means_gdf.crs)

        basin_cell_area_da = cell_area_km_da.rio.clip(basins_means_gdf[basins_means_gdf['PFAF_ID'] == basin].geometry.values, crs=basins_means_gdf.crs)
        basin_water_volume_da = ((basin_swe_da/1000) * basin_cell_area_da) # convert to km3
        basin_median_water_volume_da = basin_water_volume_da.median(dim='water_year')
        basin_water_volume_anomaly_da = basin_water_volume_da - basin_median_water_volume_da


    #   basin_swe_sum_da = basin_swe_da.sum(dim='water_year') some sort of sum and median sum (total water from snow)

        basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, 'pct_precip_as_snow'] = basin_pct_precip_as_snow_da.where(lambda x: x>0).mean().values
        basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, 'swe_median'] = basin_swe_median_da.mean().values
        basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, 'median_total_water_equivalent_km3'] = basin_median_water_volume_da.mean().values


        for water_year in water_years:
            basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, f'swe_WY{water_year}'] = basin_swe_da.sel(water_year=water_year).mean().values
            basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, f'swe_anomaly_WY{water_year}'] = basin_swe_anomaly_da.sel(water_year=water_year).mean().values
            basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, f'swe_pct_norm_WY{water_year}'] = basin_swe_pct_norm_da.sel(water_year=water_year).mean().values

            basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, f'total_water_equivalent_km3_WY{water_year}'] = basin_water_volume_da.sel(water_year=water_year).mean().values
            basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, f'total_water_equivalent_anomaly_km3_WY{water_year}'] = basin_water_volume_anomaly_da.sel(water_year=water_year).mean().values
            basins_means_gdf.loc[basins_means_gdf['PFAF_ID'] == basin, f'total_water_equivalent_pct_norm_WY{water_year}'] = (basin_water_volume_da.sel(water_year=water_year)/basin_median_water_volume_da).mean().values
    except Exception as e:
        print(f"Error processing basin {basin}: {e}")
        continue

In [ ]:
basins_means_gdf.plot(column='swe_median',legend=True, cmap='Blues', legend_kwds={'label': "SWE median [m]"},vmin=0,vmax=1)

In [ ]:
basins_means_gdf.plot(column='pct_precip_as_snow',legend=True, cmap='Purples', legend_kwds={'label': "Percent of precip that falls as snow [%]"},vmin=0,vmax=100)

In [ ]:
f,ax=plt.subplots(figsize=(12,7))
basins_means_gdf.plot.scatter(ax=ax,x='POPULATION',y='pct_precip_as_snow',c='runoff_onset_mad',cmap='Reds',edgecolor='black',linewidth=0.5)
# make x axis log scale
ax.set_xscale('log')

In [ ]:
f,axs=plt.subplots(nrows=5,ncols=2, figsize=(10,10), subplot_kw={'projection': ccrs.PlateCarree()}, dpi=300, layout='constrained',sharex=True, sharey=True)

for water_year, ax in zip(water_years,axs.flat):
    basins_means_gdf.plot(ax=ax,column=f'swe_anomaly_WY{water_year}',cmap='RdBu',vmin=-0.5,vmax=0.5,legend=False, transform=ccrs.PlateCarree())
    ax.set_title(f'{water_year}')
    gl = ax.gridlines(draw_labels=True, dms=True, x_inline=False, y_inline=False, xlocs=[-180, -120, -60, 0, 60, 120, 180], ylocs=[-60, -40, -20, 0, 20, 40, 60, 80], linestyle='--', linewidth=0.5)
    gl.top_labels=False
    gl.bottom_labels=True
    gl.right_labels=False
    gl.left_labels=True

    ax.set_extent([-180, 180, -60, 90], crs=ccrs.PlateCarree())

    ax.add_feature(cfeature.LAND, facecolor='lightgrey')
    ax.add_feature(cfeature.OCEAN, facecolor='powderblue') # 'lightcyan'
    ax.add_feature(cfeature.COASTLINE, linewidth=0.5, edgecolor='black')

In [ ]:
#basins_means_gdf.explore(column='swe_median',vmin=0,vmax=2)

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
basins_means_gdf.plot.scatter(ax=ax,y='swe_median',x='runoff_onset_median',c='runoff_onset_mad',cmap='Reds',vmin=0,vmax=30,alpha=1, s=10)
ax.set_ylim(0,3)
ax.set_xlabel("10-year median runoff onset date [DOWY]")
ax.set_ylabel("10-year median April 1st / October 1st SWE [m]")

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
basins_means_gdf.plot.scatter(ax=ax,y='swe_median',x='runoff_onset_mad',c='runoff_onset_median', cmap='viridis',s=10)
ax.set_ylim(0,3)
ax.set_xlabel("10-year runoff onset median absolute deivations [days]")
ax.set_ylabel("10-year median April 1st / October 1st SWE [m]")
# which basins have a high SWE and low MAD?
# which basins have a high SWE and high MAD (more dangerous)

In [ ]:
# f,ax=plt.subplots(figsize=(12,7))
# basins_means_gdf['POPULATION'].hist(ax=ax,bins=300)
# ax.set_ylim(0,100)

In [ ]:
# basins_means_gdf[basins_means_gdf['POPULATION'] == 0].explore()

In [ ]:
sizes = np.log(basins_means_gdf['POPULATION']+1)
sizes

In [ ]:
f,ax=plt.subplots(figsize=(12,7))
sizes.hist(ax=ax,bins=300)
#ax.set_ylim(0,100)

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
basins_means_gdf.plot.scatter(ax=ax,y='swe_median',x='runoff_onset_mad',c='runoff_onset_median', cmap='viridis',s=5*sizes)
ax.set_ylim(0,3) # 0.1,0.5,1
ax.set_xlabel("10-year runoff onset median absolute deivations [days]")
ax.set_ylabel("10-year median April 1st / October 1st SWE [m]")
# which basins have a high SWE and low MAD?
# which basins have a high SWE and high MAD (more dangerous)

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
basins_means_gdf.plot.scatter(ax=ax,y='swe_median',x='runoff_onset_median',c='runoff_onset_mad',cmap='Reds',vmin=0,vmax=30,alpha=1, s=sizes)
ax.set_ylim(0,3)
ax.set_xlabel("10-year median runoff onset date [DOWY]")
ax.set_ylabel("10-year median April 1st / October 1st SWE [m]")

In [ ]:
# maybe focus on population on x axis????
# 
# f,ax=plt.subplots(figsize=(10,10))
basins_means_gdf.plot.scatter(ax=ax,y='swe_median',x='runoff_onset_median',c='runoff_onset_mad',cmap='Reds',vmin=0,vmax=30,alpha=1, s=sizes)
ax.set_ylim(0,3)
ax.set_xlabel("10-year median runoff onset date [DOWY]")
ax.set_ylabel("10-year median April 1st / October 1st SWE [m]")

In [ ]:
basins_means_gdf.columns

In [ ]:
f,ax=plt.subplots(figsize=(10,10))
basins_means_gdf.plot.scatter(ax=ax,y='median_total_water_equivalent_km3',x='runoff_onset_mad',c='runoff_onset_median', cmap='viridis',s=5*sizes)
ax.set_xlabel("10-year runoff onset median absolute deivations [days]")
ax.set_ylabel("10-year median April 1st / October 1st total water equivalent [km^3]")
# which basins have a high SWE and low MAD?
# which basins have a high SWE and high MAD (more dangerous)

In [ ]:
basins_means_gdf['swe_median'].describe()

In [ ]:
scatter_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
f,ax=plt.subplots(figsize=(10,10))

for water_year, color in zip(water_years,scatter_colors):
    basins_means_gdf[basins_means_gdf['swe_median']>0.0].plot.scatter(ax=ax,y=f'swe_anomaly_WY{water_year}',x=f'runoff_onset_anomaly_WY{water_year}',color=color,s=1,alpha=1,label=water_year)



ax.axvline(x=0, color='black', linestyle='--')
ax.axhline(y=0, color='black', linestyle='--')

ax.set_ylim(-0.5,0.5)
ax.set_xlim(-30,30)

ax.set_xlabel("Runoff onset anomaly [days]")
ax.set_ylabel("SWE anomaly [m]")

ax.legend()


In [ ]:
scatter_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
f,ax=plt.subplots(figsize=(10,10))

for water_year, color in zip(water_years,scatter_colors):
    basins_means_gdf[basins_means_gdf['swe_median']>0.0].plot.scatter(ax=ax,y=f'total_water_equivalent_anomaly_km3_WY{water_year}',x=f'runoff_onset_anomaly_WY{water_year}',color=color,s=1,alpha=1,label=water_year)



ax.axvline(x=0, color='black', linestyle='--')
ax.axhline(y=0, color='black', linestyle='--')

#ax.set_ylim(-0.5,0.5)
ax.set_xlim(-30,30)

ax.set_xlabel("Runoff onset anomaly [days]")
ax.set_ylabel("Basin total water volume anomaly [km^3]")

ax.legend()

In [ ]:
# Define water years
water_years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Create empty list to store data rows
data_rows = []

# Filter to basins with SWE median > 0.0 as in your previous analyses
filtered_data = basins_means_gdf[basins_means_gdf['swe_median'] > 0.0]

# Loop through each basin
for idx, basin_row in filtered_data.iterrows():
    basin_id = basin_row['PFAF_ID']  # Using PFAF_ID as basin identifier
    
    # For each water year, extract the anomaly values
    for year in water_years:
        runoff_col = f'runoff_onset_anomaly_WY{year}'
        swe_col = f'swe_anomaly_WY{year}'
        
        # Check if columns exist and values are not NaN
        if runoff_col in filtered_data.columns and swe_col in filtered_data.columns:
            runoff_anomaly = basin_row[runoff_col] 
            swe_anomaly = basin_row[swe_col]
            
            # Only include rows where both values are valid
            if pd.notna(runoff_anomaly) and pd.notna(swe_anomaly):
                data_rows.append({
                    'basin': basin_id,
                    'runoff_onset_anomaly': runoff_anomaly,
                    'swe_anomaly': swe_anomaly,
                    'water_year': year
                })

# Create dataframe from collected data
anomalies_df = pd.DataFrame(data_rows)
anomalies_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Create figure for KDE plot
fig_kde, ax_kde = plt.subplots(figsize=(10, 10))

# Create the KDE plot
sns.kdeplot(
    data=anomalies_df,
    x='runoff_onset_anomaly',
    y='swe_anomaly',
    fill=True,
    cmap="viridis",
    levels=30,
    ax=ax_kde
)

# Calculate line of best fit
slope, intercept, r_value, p_value, std_err = stats.linregress(
    anomalies_df['runoff_onset_anomaly'].dropna(), 
    anomalies_df['swe_anomaly'].dropna()
)

# Add line of best fit to KDE plot
x_line = np.linspace(-30, 30, 100)
y_line = slope * x_line + intercept
ax_kde.plot(x_line, y_line, color='red', linestyle='--', linewidth=2, 
            label=f'y = {slope:.4f}x + {intercept:.4f} (r = {r_value:.2f})')

# Add reference lines
ax_kde.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax_kde.axhline(y=0, color='black', linestyle='--', alpha=0.5)

# Set limits and labels
ax_kde.set_ylim(-0.3, 0.3)
ax_kde.set_xlim(-30, 30)
ax_kde.set_xlabel('Runoff Onset Anomaly [days]')
ax_kde.set_ylabel('SWE Anomaly [m]')
ax_kde.set_title('KDE Plot of SWE vs Runoff Onset Anomalies')
ax_kde.legend()

# Create figure for hexbin plot
fig_hex, ax_hex = plt.subplots(figsize=(10, 10))

# Create the hexbin plot
hb = ax_hex.hexbin(
    anomalies_df['runoff_onset_anomaly'], 
    anomalies_df['swe_anomaly'], 
    gridsize=100, 
    cmap='viridis', 
    mincnt=1,
    bins='log'  # Use log scale for better visualization
)

# Add line of best fit to hexbin plot
ax_hex.plot(x_line, y_line, color='red', linestyle='--', linewidth=2,
            label=f'y = {slope:.4f}x + {intercept:.4f} (r = {r_value:.2f})')

# Add reference lines
ax_hex.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax_hex.axhline(y=0, color='black', linestyle='--', alpha=0.5)

# Set limits and labels
ax_hex.set_ylim(-0.3, 0.3)
ax_hex.set_xlim(-30, 30)
ax_hex.set_xlabel('Runoff Onset Anomaly [days]')
ax_hex.set_ylabel('SWE Anomaly [m]')
ax_hex.set_title('Hexbin Plot of SWE vs Runoff Onset Anomalies')
ax_hex.legend()

# Add colorbar to hexbin plot
cb = fig_hex.colorbar(hb, ax=ax_hex)
cb.set_label('log10(count)')

plt.tight_layout()
plt.show()

In [ ]:
scatter_colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf']
f,ax=plt.subplots(figsize=(10,10))

for water_year, color in zip(water_years,scatter_colors):
    basins_means_gdf.plot.scatter(ax=ax,y=f'swe_pct_norm_WY{water_year}',x=f'runoff_onset_anomaly_WY{water_year}',color=color,s=1, alpha=1)

ax.set_ylim(0,1000)
ax.set_xlim(-60,60)

# ax.set_ylim(0)
# ax.set_xlim(-30,30)

ax.axvline(x=0, color='black', linestyle='--')
ax.axhline(y=100, color='black', linestyle='--')

In [ ]:
basins_means_gdf

In [ ]:
# Define water years
water_years = [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

# Create empty list to store data rows
data_rows = []

# Filter to basins with SWE median > 0.0 as in your previous analyses
filtered_data = basins_means_gdf[basins_means_gdf['swe_median'] > 0.0]

# Loop through each basin
for idx, basin_row in filtered_data.iterrows():
    basin_id = basin_row['PFAF_ID']  # Using PFAF_ID as basin identifier
    
    # For each water year, extract the anomaly values
    for year in water_years:
        runoff_col = f'runoff_onset_anomaly_WY{year}'
        swe_col = f'total_water_equivalent_anomaly_km3_WY{year}'
        
        # Check if columns exist and values are not NaN
        if runoff_col in filtered_data.columns and swe_col in filtered_data.columns:
            runoff_anomaly = basin_row[runoff_col] 
            swe_anomaly = basin_row[swe_col]
            
            # Only include rows where both values are valid
            if pd.notna(runoff_anomaly) and pd.notna(swe_anomaly):
                data_rows.append({
                    'basin': basin_id,
                    'runoff_onset_anomaly': runoff_anomaly,
                    'basin_total_water_volume_anomaly': swe_anomaly,
                    'water_year': year
                })

# Create dataframe from collected data
anomalies_df = pd.DataFrame(data_rows)
anomalies_df

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# Create figure for KDE plot
fig_kde, ax_kde = plt.subplots(figsize=(10, 10))

# Create the KDE plot
sns.kdeplot(
    data=anomalies_df,
    x='runoff_onset_anomaly',
    y='basin_total_water_volume_anomaly',
    fill=True,
    cmap="viridis",
    levels=30,
    ax=ax_kde
)

# Calculate line of best fit
slope, intercept, r_value, p_value, std_err = stats.linregress(
    anomalies_df['runoff_onset_anomaly'].dropna(), 
    anomalies_df['basin_total_water_volume_anomaly'].dropna()
)

# Add line of best fit to KDE plot
x_line = np.linspace(-30, 30, 100)
y_line = slope * x_line + intercept
ax_kde.plot(x_line, y_line, color='red', linestyle='--', linewidth=2, 
            label=f'y = {slope:.4f}x + {intercept:.4f} (r = {r_value:.2f})')

# Add reference lines
ax_kde.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax_kde.axhline(y=0, color='black', linestyle='--', alpha=0.5)

# Set limits and labels
#ax_kde.set_ylim(-0.3, 0.3)
ax_kde.set_xlim(-30, 30)
ax_kde.set_xlabel('Runoff Onset Anomaly [days]')
ax_kde.set_ylabel('Basin total water volume anomaly [km^3]')
ax_kde.set_title('KDE Plot of Basin Total Water Volume Anomalies vs Runoff Onset Anomalies')
ax_kde.legend()

# Create figure for hexbin plot
fig_hex, ax_hex = plt.subplots(figsize=(10, 10))

# Create the hexbin plot
hb = ax_hex.hexbin(
    anomalies_df['runoff_onset_anomaly'], 
    anomalies_df['basin_total_water_volume_anomaly'], 
    gridsize=100, 
    cmap='viridis', 
    mincnt=1,
    bins='log'  # Use log scale for better visualization
)

# Add line of best fit to hexbin plot
ax_hex.plot(x_line, y_line, color='red', linestyle='--', linewidth=2,
            label=f'y = {slope:.4f}x + {intercept:.4f} (r = {r_value:.2f})')

# Add reference lines
ax_hex.axvline(x=0, color='black', linestyle='--', alpha=0.5)
ax_hex.axhline(y=0, color='black', linestyle='--', alpha=0.5)

# Set limits and labels
#ax_hex.set_ylim(-0.3, 0.3)
ax_hex.set_xlim(-30, 30)
ax_hex.set_xlabel('Runoff Onset Anomaly [days]')
ax_hex.set_ylabel('Basin total water volume anomaly [km^3]')
ax_hex.set_title('Hexbin Plot of Basin Total Water Volume Anomalies vs Runoff Onset Anomalies')
ax_hex.legend()

# Add colorbar to hexbin plot
cb = fig_hex.colorbar(hb, ax=ax_hex)
cb.set_label('log10(count)')

plt.tight_layout()
plt.show()

In [ ]:
# f,ax=plt.subplots(nrows=5,ncols=2,figsize=(15,20),subplot_kw={'projection': ccrs.PlateCarree()}) # ccrs.Robinson()
# for water_year, ax in zip(weighted_mean_filtered_ds.water_year.values, ax.flat):
#     basins_counts_gdf.plot(ax=ax,column=f'runoff_onset_anomaly_WY{water_year}',transform=ccrs.PlateCarree(),norm=colors.LogNorm())
#     ax.set_title(f'{water_year}')
#     ax.coastlines()

# f.suptitle('Runoff Onset Anomaly runoff onset pixel counts by River Basin')
# f.tight_layout()

# f,ax=plt.subplots(nrows=5,ncols=2,figsize=(15,20),subplot_kw={'projection': ccrs.PlateCarree()}) # ccrs.Robinson()
# for water_year, ax in zip(weighted_mean_filtered_ds.water_year.values, ax.flat):
#     basins_areas_gdf.plot(ax=ax,column=f'runoff_onset_anomaly_WY{water_year}',transform=ccrs.PlateCarree())
#     ax.set_title(f'{water_year}')
#     ax.coastlines()

# f.suptitle('Runoff Onset Anomaly runoff onset pixel areas by River Basin')
# f.tight_layout()

# f,ax=plt.subplots(nrows=5,ncols=2,figsize=(15,20),subplot_kw={'projection': ccrs.PlateCarree()}) # ccrs.Robinson()
# for water_year, ax in zip(weighted_mean_filtered_ds.water_year.values, ax.flat):
#     basins_pct_areas_gdf.plot(ax=ax,column=f'runoff_onset_anomaly_WY{water_year}',transform=ccrs.PlateCarree(),vmin=0,vmax=100,cmap='gist_stern') # cmap='rainbow'
#     ax.set_title(f'{water_year}')
#     ax.coastlines()

# f.suptitle('Runoff Onset Anomaly pct area with runoff onset pixels by River Basin')
# f.tight_layout()

In [ ]:
# Load and prepare world data
